# Page 9 — "Outcome vs Process Supervision": two graders on the same chains

One set of reasoning chains, graded twice. An **outcome** grader reads the final answer. A
**process** grader reads every step. This notebook measures what the second one buys, in three
numbers the page has never been able to state:

1. **Detection** — among chains that reached the *right* answer, how many contain a step that is
   demonstrably false. Outcome grading passes every one of them, and page 12's STaR filter would
   fine-tune on every one of them.
2. **Localisation** — among chains that reached the *wrong* answer, how much of the chain was
   sound up to the break. Outcome grading condemns all of it.
3. **Selection** — a process-aware rung on page 8's ladder, if page 8's inputs are on disk.

**Two process graders, and one of them has to earn its labels.** The arithmetic checker is
deterministic: where it judges it is right by construction, and a reader can check any label by
hand. `Qwen2.5-Math-PRM-7B` is a model, so its labels are opinions until measured. Section 6
measures them against the checker on the steps both can see, reports AUC and specificity rather
than claiming a reliability, and **withholds the PRM's rows entirely if they miss a bar set in
section 2 before the run**. That is the move page 8 makes with its verifiers, with one addition:
a verifier that fails its own measurement does not get a footnote, it gets suppressed.

Detection and localisation are the checker's alone, so the page's two load-bearing numbers never
depend on whether the PRM showed up or how it did.

Writes `out/page_09/`: `traces.jsonl` (chains with per-step labels), `derived.json`,
`manifest.json`.

All rationale lives in [`docs/page_9_refactor.md`](../docs/page_9_refactor.md).

## How to run this

| Mode | Chains | Process grader | Needs | Time |
|---|---|---|---|---|
| `REPLAY` | page 7's pile, filtered (~1,600) | checker only | no GPU, no weights | under a minute |
| `QUICK` | 24 generated here | checker only | a laptop CPU | ~35 min |
| `FULL` | page 7's pile if present | checker + `Qwen2.5-Math-PRM-7B` | a 16 GB GPU | ~4 min |

Those times are measured on an RTX 5080 with the weights already cached, `QUICK`'s excepted, which
is an estimate for a CPU. They are worth stating because an earlier version of this table guessed,
and guessed high by two orders of magnitude: it claimed `REPLAY` spent about fifteen minutes
in sympy. It does not. The checker calls sympy only on steps that are closed-form numeric on both sides,
which is 6% of them, so all 1,580 chains grade in about two seconds and the whole of `REPLAY` runs
in ten. There is no model to load and nothing to download.

`FULL` adds one PRM forward pass per chain, about 2.6 minutes, and half a minute to quantise 7.6B
weights to 8-bit. A first `FULL` run also fetches 15.3 GB of PRM weights. `FISH_SPECIMENS` is off
by default and adds about ten minutes when it is on; section 10 says when that is worth it.

**This page grades only chains the model ended itself**, which is page 7's pile where the cap
never bound. A chain we cut off and forced to commit is broken by *us*, so counting that break as
the model reasoning badly would report our own experiment as a finding. Section 3 applies the
filter and measures what it dropped; page 7's pile was sampled, not greedy, so section 3 also
records the source's real decoding rather than assuming this notebook's.

The generator is `Qwen2.5-3B-Instruct`, Act 2's shared instrument, so page 9 describes the same
population as the pages before it. The process graders are a deterministic arithmetic checker
and, in `FULL`, `Qwen2.5-Math-PRM-7B`; section 5 states what that PRM is and section 6 measures
it.


In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (small, generates), "FULL"
SEED = 123                # selects the items, and it is page 7's seed so the draws match

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"      # pages 6 and 7's primary, so the pages compare
# Same Qwen2.5 lineage as the generator, math-specialised branch. NOT a different
# family, and section 5 says what that costs us.
PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"

PRESETS = {
    "REPLAY": dict(n_items=100, use_prm=False, may_generate=False),
    "QUICK":  dict(n_items=24,  use_prm=False, may_generate=True),
    "FULL":   dict(n_items=100, use_prm=True,  may_generate=True),
}
CFG = PRESETS[RUN_MODE]
USE_PRM = CFG["use_prm"]

# Page 6's top rung. A safety cap, not a budget: a chain that hits it is one WE cut,
# and section 3 drops those rather than grading them.
CHAIN_CAP = 2048

# Where a step ends. Passed at every call site because it is a claim, not a default
# (harness.step_spans has none). Same value as page 5, so a chain segmented here and
# a chain segmented there break the same way.
MIN_STEP_CHARS = 12

# ---- what the PRM has to clear before the page may quote it ----------------
# 0.5 is a starting point, not a threshold; section 6 fits the operating point on the
# checker's labels and records it as fitted.
PRM_THRESHOLD_START = 0.5
PRM_FIT_THRESHOLD = True

# Youden's J is nearly flat near its maximum on these labels, so the argmax is decided
# by differences too small to mean anything, and two identical runs chose cut-offs 0.057
# apart. Every candidate within this much of the best J counts as tied, and section 6
# breaks the tie on specificity rather than on floating-point luck.
PRM_J_TOLERANCE = 0.01

# 7.6B in bf16 (~15 GB) does not fit beside its activations on a 16 GB card; 8-bit
# does, at a cost to the scores that section 6 measured. Set to None on a bigger card.
PRM_QUANT = "8bit"

# Stated in advance so the gate is a pre-registration, not a post-hoc judgement; section 5
# says why each bar. A known-false step must score at least PRM_MIN_SEPARATION below a true
# one in the preflight; the right sign is not enough, since a broken loader produced it at 0.002.
PRM_MIN_SEPARATION = 0.20

PRM_MIN_AUC = 0.65
PRM_MIN_SPECIFICITY = 0.80
PRM_MIN_COMPARED_STEPS = 30      # below this the reliability is too noisy to gate on

print(f"mode={RUN_MODE}  generator={PRIMARY}")
print(f"process graders: arithmetic checker" + (f" + {PRM_MODEL}" if USE_PRM else " only"))
if USE_PRM:
    print(f"the PRM publishes only if AUC >= {PRM_MIN_AUC:.2f} and specificity >= "
          f"{PRM_MIN_SPECIFICITY:.0%} over >= {PRM_MIN_COMPARED_STEPS} compared steps")

# ---- specimen fishing: a SECOND source for the worked examples -------------
# The three specimens on the page are ILLUSTRATIVE, not the measured population.
# Section 10 draws them from page 7's L4-5 pile, pinned to the GCD of 3339, 2961 and
# 1491, whose one-line Euclidean divisions a reader can mark by hand. This block is
# the other source: SHORT easy chains (levels 1-3, many samples each), graded by the
# same checker and PRM, for a role the pile cannot fill. Sections 7-9's measured
# rates stay on the L4-5 pile either way; only worked examples come from here.
#
# ON in FULL, because finding a better `lucky` chain is what a FULL run is FOR.
# The pinned GCD chain is the only one of the 5 lucky chains in page 7's 1,580 that
# has all three properties at once: a one-line claim a reader can check by hand, a
# partner on the same problem, and a PRM that scores the false step SOUND. It costs
# 13 steps to get them. This pass looks for something shorter that keeps all three.
#
# It cannot displace anything on its own. Section 10 falls back per ROLE and prefers
# the pile, so this produces a candidate list to READ; pinning a winner means setting
# SPECIMEN_PROBLEM by hand.
# Measured, and it answered its question: over 238 problems and 7,358 chains it found
# 51 lucky candidates, 28 of them real, and exactly ONE kept the property the page's
# third beat needs (checker proves the step false, PRM calls it sound) at 13 steps and
# a claim of "-1 = 0", which is no better than the pinned pair. The PRM catches exactly
# the crisp arithmetic slips that make a legible specimen, so the search is anti-
# correlated with what it was looking for. Off for a publishing run; turn it on to
# search again, and read the candidate table section 6 prints.
FISH_SPECIMENS = False                   # needs the generator; ignored in REPLAY/QUICK
FISH_LEVELS = (1, 2, 3)
FISH_N_PROBLEMS = 238                    # every L1-3 problem in MATH-500. Candidates
                                         # cluster on a few problems (8 lucky chains
                                         # over 60 problems came from just 2), so
                                         # breadth is what finds new ones
FISH_SAMPLES_PER_PROBLEM = 32            # lucky chains are rare (~0.3%), so sample hard
FISH_TEMPERATURE = 0.30                  # page 7's pile temperature, so the model behaves as before
FISH_TOP_P = 0.95
FISH_CAP = 1024                          # easy problems need far fewer tokens than L4-5
FISH_MAX_STEPS = 8                       # a specimen longer than this is not legible, and
                                         # section 10 holds the fished pools to it


In [ ]:
import json
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_09")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

P7_DIR = Path("out/page_07")
P7_TRACES = P7_DIR / "traces.jsonl"
P7_MANIFEST = P7_DIR / "manifest.json"
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median = harness.mean, harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. The outcome grader

Inherited unchanged from ch03, and it is the grader this whole page is arguing about.
`extract_final_candidate` pulls the last `\boxed{...}`; `grade_answer` checks equivalence.

Note what it can see: the last line. It has no opinion about the hundred tokens before it, which
is the entire subject of this page. The fifth check below is the page in miniature.


In [ ]:
CHECKS = [
    (r"The answer is \boxed{9}.",                        "9",    True),
    (r"27",                                              "9",    False),
    (r"\boxed{\frac{1}{2}}",                             "0.5",  True),
    (r"\boxed{1,024}",                                   "1024", True),
    # right answer via nonsense: the outcome grader says yes, which is the page's subject
    (r"Cancel the 6s: 16/64 -> 1/4. So \boxed{\frac{1}{4}}", "0.25", True),
    (r"I am not sure.",                                  "9",    False),
]

rows = [(raw, gt, exp, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, exp in CHECKS]
bad = [r for r in rows if r[3] != r[2]]
if bad:
    for raw, gt, exp, got in rows:
        print(f"{'ok ' if got == exp else 'FAIL'} {extract_final_candidate(raw)!r:<26} "
              f"vs {gt!r:<8} -> {got}")
    raise AssertionError(f"{len(bad)} outcome-grader check(s) behaved unexpectedly")

print(f"outcome grader preflight: {len(rows)}/{len(rows)} as expected")
print("the fifth case is not a bug in the grader. A right answer reached by nonsense is")
print("correct as far as an outcome check can see, and measuring how often that happens")
print("is what this page is for.")


## 2. The item set

MATH-500, levels 4 and 5, page 6's band, so this page and the ones before it describe the same
population.

Two reasons the band matters here specifically. **Wrong answers**: section 8 has nothing to
localise in without them. **Chain length**: a one-line solution cannot show a break partway
through, and easy problems get one-line solutions.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted((r for r in math500 if int(r["level"]) in (4, 5)),
              key=lambda r: r["unique_id"])          # sort before shuffling
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
ITEMS_BY_ID = {r["unique_id"]: r for r in items}
ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)

lv = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"{len(items)} items of {len(band)} available at levels 4-5   {lv}   "
      f"sha256 {ITEM_SET_HASH[:12]}")

# Page 7 asserts its draw against page 6's rather than trusting that two notebooks with
# the same constants drew the same items. Do the same here: a divergent draw would not
# fail, it would silently shrink the pool in section 3 and look like a thin run.
P7_MANIFEST_DATA = json.loads(P7_MANIFEST.read_text()) if P7_MANIFEST.exists() else None
SAME_ITEMS_AS_PAGE_7 = None
if P7_MANIFEST_DATA:
    p7_items = P7_MANIFEST_DATA.get("items", {})
    SAME_ITEMS_AS_PAGE_7 = p7_items.get("item_set_sha256") == ITEM_SET_HASH
    print(f"page 7 item set  {str(p7_items.get('item_set_sha256'))[:12]}  "
          f"n={p7_items.get('n')}  seed={p7_items.get('seed')}")
    print(f"-> {'identical draw' if SAME_ITEMS_AS_PAGE_7 else 'DIFFERENT draw from page 7'}")
    if not SAME_ITEMS_AS_PAGE_7:
        print("   Only the overlap is gradable, and the pages then describe different")
        print("   populations. Match page 7's seed and n_items, or say so on the page.")
else:
    print(f"{P7_MANIFEST} not found; the draw cannot be checked against page 7's")

## 3. Which chains this page is allowed to grade

The filter, and the one design decision here most worth understanding.

Grading a chain for bad reasoning only means something if the reasoning is the model's own. Three
sources, and only some of each qualifies:

| source | qualifies? |
|---|---|
| page 7's **pile** | yes, where `truncated` is false. These run to their natural end. |
| page 7's **split** axis | no. Each chain gets `B/N` tokens by construction. |
| page 6's sweep | no. Capped and force-committed on purpose; that is its experiment. |

So this is a filter on `truncated`, not a blanket refusal to reuse. If nothing survives it and
the mode may generate, the notebook runs its own uncapped pass instead.

**The filter is a selection, and the page has to say so rather than only the notebook.** A chain
hits the 2048-token cap because it was long, and it was long because the problem was hard, so
dropping cap-hitting chains preferentially drops hard problems and the failures attached to
them. Everything below is therefore conditional on *chains the model ended itself*, which is a
narrower population than *all chains* and skews easier. The cell measures the skew instead of
waving at it: it prints the level mix and the outcome accuracy of what survived against what was
dropped, and both land in `derived.json` so the caption can carry the real number.

### Every usable chain, not one per problem, and the reason is page 12

An earlier version graded the first chain per problem, which is a defensible single draw and
turned out to measure nothing at all: 51 correct chains, of which the checker could prove a false
step in **zero**. That is not a finding, it is an absence of statistical power, and it also left
two of section 10's three specimen roles unfillable.

The pile holds roughly sixteen independent samples per problem, and the population that matters
here is **rationales, not problems**. Page 12 is STaR: generate many rationales, keep every one
that reaches the right answer, fine-tune on all of them. The contamination rate of that filter is
therefore *of all correct rationales, how many carry a false step*, which is a rate over chains.
Grading one chain per problem would answer a question page 12 does not ask.

**What that costs, and it is paid rather than hidden.** Sixteen chains from one problem are not
sixteen independent observations, so the **unit of independence is the problem**. Section 7
reports the chain-level rate the STaR framing wants *and* the number of distinct problems behind
it, and a reader can see how much of the rate rests on how few problems. It costs less time than
it sounds like: this is roughly 1,600 chains through sympy rather than 100, and the whole pass
takes about two seconds, because the checker reaches sympy only on the steps that are closed-form
numeric on both sides.


In [ ]:
def usable(rec):
    """A chain whose ending was the model's choice, not ours."""
    return (not rec.get("truncated")) and not rec.get("forced") and rec.get("tokens")


def pick_pool_config(recs):
    """Name page 7's pile from its manifest, or recover it from the records.

    Page 7's dump holds several configurations and only ONE of them qualifies: the
    split axis is budget-starved by construction, the diversity family sits at
    temperatures the pile does not, and the temperature-0 configuration is one chain
    replicated. Pooling them would grade four decoding regimes at once and then call
    the mixture "the generator", which is the provenance defect this cell exists to
    avoid. Prefer the manifest; the fallback is the full-cap sampled configuration
    with the most records, and it says out loud that it guessed.
    """
    named = (P7_MANIFEST_DATA or {}).get("pile", {}).get("config_id")
    if named:
        return named, "page 7's manifest"
    at_cap = [r for r in recs
              if r.get("chain_budget") == CHAIN_CAP and (r.get("temperature") or 0) > 0]
    if not at_cap:
        return None, None
    guess = Counter(r["config_id"] for r in at_cap).most_common(1)[0][0]
    print(f"page 7's manifest is absent; inferring the pile as {guess!r} from the records")
    return guess, "inferred from the records"


source, records = None, []
CHAIN_PROVENANCE, FILTER_AUDIT = None, None

if P7_TRACES.exists():
    latest = {}
    for line in P7_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            latest[(r["run_id"], r.get("config_id"), r["problem_id"],
                    r.get("sample_index", 0))] = r
    raw7 = list(latest.values())
    P7_RUN_ID = Counter(r["run_id"] for r in raw7).most_common(1)[0][0]
    POOL_CONFIG, POOL_CONFIG_FROM = pick_pool_config(
        [r for r in raw7 if r["run_id"] == P7_RUN_ID])

    pool = [r for r in raw7
            if r["run_id"] == P7_RUN_ID
            and r.get("config_id") == POOL_CONFIG
            and r.get("model_id", PRIMARY) == PRIMARY
            and r["problem_id"] in ITEMS_BY_ID]
    kept = sorted([r for r in pool if usable(r)],
                  key=lambda r: (r["problem_id"], r.get("sample_index", 0)))
    dropped = [r for r in pool if not usable(r)]
    by_pid = {}
    for r in kept:
        by_pid.setdefault(r["problem_id"], []).append(r)

    print(f"page 7 run {P7_RUN_ID}")
    print(f"  pile config {POOL_CONFIG!r} ({POOL_CONFIG_FROM})")
    print(f"  {len(pool):,} records on our items, {len(kept):,} ran to their natural end "
          f"({len(kept)/max(len(pool),1):.0%}),")
    print(f"  covering {len(by_pid)} of {len(items)} problems")

    if pool:
        # What the filter costs us, measured rather than asserted. If the dropped
        # chains are markedly harder or markedly more often wrong, sections 7 and 8
        # describe an easier population than "this model on MATH-500 L4-5".
        def _mix(rs):
            return "+".join(f"L{n}x{c}" for n, c in
                            sorted(Counter(int(r.get("level", 0)) for r in rs).items()))

        FILTER_AUDIT = {
            "n_records": len(pool), "n_kept": len(kept), "n_dropped": len(dropped),
            "kept_outcome_accuracy": mean([r["correct"] for r in kept]) if kept else None,
            "dropped_outcome_accuracy": (mean([r["correct"] for r in dropped])
                                         if dropped else None),
            "kept_level_mix": _mix(kept), "dropped_level_mix": _mix(dropped),
            "n_problems_covered": len(by_pid), "n_problems_asked": len(items),
        }
        harness.table(
            "what the natural-ending filter selected",
            ["chains", "n", "levels", "outcome accuracy"],
            [["the model ended them", len(kept), FILTER_AUDIT["kept_level_mix"],
              f"{FILTER_AUDIT['kept_outcome_accuracy']:.0%}" if kept else "n/a"],
             ["we cut them at the cap", len(dropped), FILTER_AUDIT["dropped_level_mix"],
              f"{FILTER_AUDIT['dropped_outcome_accuracy']:.0%}" if dropped else "n/a"]],
            note="everything below is conditional on the first row, which is the easier "
                 "of the two populations")

    if len(by_pid) >= max(8, CFG["n_items"] // 4):
        source = "page_07_pile"
        first = kept[0]
        per_problem = [len(v) for v in by_pid.values()]
        CHAIN_PROVENANCE = {
            "source": source,
            "page_07_run_id": P7_RUN_ID,
            "page_07_config_id": POOL_CONFIG,
            "config_id_from": POOL_CONFIG_FROM,
            "decoding": "sampled",
            "temperature": first.get("temperature"),
            "top_p": first.get("top_p"),
            "chain_budget": first.get("chain_budget"),
            "prompt_template": (P7_MANIFEST_DATA or {}).get("prompt_template"),
            "prompt_suffix": None,     # page 7's chains never saw this page's suffix
            "chain_per_problem": f"every usable chain, {min(per_problem)}-{max(per_problem)} "
                                 f"per problem (median {median(per_problem):.0f})",
            "unit_of_independence": "the problem, not the chain",
        }
        print(f"\n  decoding: sampled at T={CHAIN_PROVENANCE['temperature']}, "
              f"top_p={CHAIN_PROVENANCE['top_p']}, cap {CHAIN_PROVENANCE['chain_budget']}")
        print("  These chains were NOT generated with this notebook's step-per-line")
        print("  suffix, and the manifest records that rather than implying otherwise.")
        records = [dict(r, question=ITEMS_BY_ID[r["problem_id"]]["problem"],
                        ground_truth=ITEMS_BY_ID[r["problem_id"]]["answer"],
                        outcome_correct=bool(r["correct"])) for r in kept]
    else:
        print("        too few to stand on; generating instead")
else:
    print(f"{P7_TRACES} not found")

# The download has to be re-derivable on its own. `traces.jsonl` beside this notebook
# IS the graded corpus: page 7's chains, already filtered, carrying the fields sections
# 4 to 11 need. Every label below is recomputed from the trace text, so replaying from
# it reproduces the run rather than trusting its stored labels.
# The download ships flat, so the file may sit beside the notebook rather than under
# out/page_09/. Accept either: README.md tells a reader to move it, and not needing to
# is better than being told.
PUBLISHED = next((q for q in (TRACES_PATH, Path("traces.jsonl")) if q.exists()), None)

if not records and PUBLISHED is not None:
    own = [json.loads(l) for l in PUBLISHED.open(encoding="utf-8") if l.strip()]
    own = [r for r in own if usable(r) and r["problem_id"] in ITEMS_BY_ID]
    if own:
        source = "page_09_published_traces"
        first = own[0]
        by_pid = {}
        for r in own:
            by_pid.setdefault(r["problem_id"], []).append(r)
        per_problem = [len(v) for v in by_pid.values()]
        CHAIN_PROVENANCE = {
            "source": source,
            "page_07_run_id": first.get("run_id"),
            "page_07_config_id": first.get("config_id"),
            "config_id_from": "carried on the published records",
            "decoding": "sampled" if (first.get("temperature") or 0) > 0 else "greedy",
            "temperature": first.get("temperature"),
            "top_p": first.get("top_p"),
            "chain_budget": first.get("chain_budget"),
            "prompt_template": None,
            "prompt_suffix": None,
            "chain_per_problem": f"every usable chain, {min(per_problem)}-{max(per_problem)} "
                                 f"per problem (median {median(per_problem):.0f})",
            "unit_of_independence": "the problem, not the chain",
        }
        print(f"{P7_TRACES} is absent, so replaying the {len(own):,} chains in {PUBLISHED}")
        print(f"beside this notebook, over {len(by_pid)} problems. Same chains, same filter;")
        print("every label is recomputed here rather than read from the file.")
        records = [dict(r, question=ITEMS_BY_ID[r["problem_id"]]["problem"],
                        ground_truth=ITEMS_BY_ID[r["problem_id"]]["answer"],
                        outcome_correct=bool(r["correct"])) for r in own]
        FILTER_AUDIT = FILTER_AUDIT or {
            "n_records": len(own), "n_kept": len(own), "n_dropped": 0,
            "kept_outcome_accuracy": mean([r["correct"] for r in own]),
            "dropped_outcome_accuracy": None,
            "kept_level_mix": "+".join(
                f"L{n}x{k}" for n, k in
                sorted(Counter(int(r.get("level", 0)) for r in own).items())),
            "dropped_level_mix": "",
            "n_problems_covered": len(by_pid), "n_problems_asked": len(items),
            "note": "replayed from the published traces, which were already filtered",
        }

# Whether these chains describe page 7's population, and there are two ways to know.
# The strict one compares item-set hashes against page 7's manifest. The download has
# no manifest to compare against, but it does not need one: its chains ARE page 7's and
# carry its run id, and section 3 only grades problems that are in this notebook's draw.
# Establishing the same fact by provenance is not a weaker claim, it is the same claim
# with a shorter proof, and the manifest records which route was taken.
SAME_POPULATION = bool(
    SAME_ITEMS_AS_PAGE_7
    or (source in ("page_07_pile", "page_09_published_traces")
        and (CHAIN_PROVENANCE or {}).get("page_07_run_id")))
POPULATION_EVIDENCE = (
    "item-set hash matches page 7's manifest" if SAME_ITEMS_AS_PAGE_7
    else "the chains carry page 7's run id and were filtered to this notebook's draw"
    if SAME_POPULATION else "not established")
if records:
    print(f"same population as page 7: {SAME_POPULATION} ({POPULATION_EVIDENCE})")

if not records and not CFG["may_generate"]:
    raise RuntimeError(
        f"REPLAY has no chains to grade. It looks for page 7's pile at {P7_TRACES}, then for "
        f"published traces at {TRACES_PATH}, and found neither. Run "
        f"page_07_frequency_voting.ipynb, keep this notebook beside the traces.jsonl it "
        f"shipped with, or use QUICK to generate a small pass here.")

print(f"\nsource: {source or 'this notebook (see the next cell)'}")


In [ ]:
COT_SUFFIX = "\n\nSolve this step by step, one step per line."

if not records:
    device = get_device()
    dtype = harness.pick_dtype(device)
    loaded = harness.load_model(PRIMARY, device=device, dtype=dtype)
    model, tok = loaded["model"], loaded["tokenizer"]
    eos_ids = harness.end_of_turn_ids(tok)

    store = harness.TraceStore(TRACES_PATH, run_id=f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
                               key_fields=("problem_id",))
    store.report()

    for row in harness.progress(items, desc="generating", unit="item"):
        hit = store.get(row["unique_id"])
        if hit:
            records.append(hit)
            continue
        g = harness.generate(render_prompt(row["problem"]) + COT_SUFFIX, model, tok,
                             enable_thinking=False, max_new_tokens=CHAIN_CAP, eos_ids=eos_ids)
        extracted = extract_final_candidate(g["text"])
        records.append(store.append({
            "problem_id": row["unique_id"], "level": int(row["level"]),
            "subject": row["subject"], "question": row["problem"],
            "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
            "tokens_generated": g["tokens_generated"], "truncated": g["truncated"],
            "extracted_answer": extracted,
            "outcome_correct": bool(grade_answer(extracted, row["answer"])),
        }))

    cut = [r for r in records if r["truncated"]]
    if cut:
        print(f"\ndropping {len(cut)} chain(s) the cap cut: a chain WE ended is not a chain")
        print("the model got wrong, and grading it would report our cap as its error.")
        records = [r for r in records if not r["truncated"]]

    source = "generated_here"
    # A different decoding regime from page 7's pile and a different prompt, so it is
    # recorded as its own thing. A run that mixed the two sources and described them
    # with one line would be reporting a population that never existed.
    CHAIN_PROVENANCE = {
        "source": source,
        "decoding": "greedy",
        "temperature": 0.0, "top_p": None,
        "chain_budget": CHAIN_CAP,
        "prompt_template": render_prompt("{question}"),
        "prompt_suffix": COT_SUFFIX,
        "chain_per_problem": "the one chain generated per problem",
    }
    del model, tok, loaded            # the caller's references are what keep it alive
    harness.free_memory()

n_right = sum(r["outcome_correct"] for r in records)
print(f"\n{len(records)} chains: {n_right} right, {len(records) - n_right} wrong "
      f"({n_right/max(len(records),1):.0%} by the outcome grader)")
print(f"decoding: {CHAIN_PROVENANCE['decoding']}"
      + (f" at T={CHAIN_PROVENANCE['temperature']}"
         if CHAIN_PROVENANCE["decoding"] == "sampled" else ""))

## 4. The arithmetic checker

Segmentation comes from `harness.step_spans`, the same rule pages 5 and 7 use, so a chain broken
into steps here breaks the same way there. The checker itself stays in this notebook, because it
is a claim rather than plumbing and a reader has to be able to read it.

### The precision rule, which is the whole design

The checker may judge **only claims that are closed-form numeric on both sides, with no symbols.**
`3/4 = 9/12` is checkable. `Let x = 5` is a definition, not a false claim. `f(2) = 7` depends on a
definition made earlier in the chain. A checker that flagged those as false would be worse than no
checker at all, because the one thing this one has going for it is that its labels are facts a
reader can verify by hand.

Every step therefore gets one of three labels, and the third is not a hole to hide:

| label | meaning |
|---|---|
| `true` | every numeric claim in the step checks out |
| `false` | at least one numeric claim is arithmetically wrong |
| `unknown` | nothing in the step is a closed-form numeric claim |

High precision, low recall, stated openly.

### Finding the claim, which is where the recall actually went

The precision rule is the easy half. The hard half is that a rule this strict, pointed at real
output, can end up judging almost nothing, and a checker that judges nothing is not cautious, it
is broken. On the first run against this model's chains it labelled **1 step out of 1,192**. Two
changes took that to about 7%, and neither loosens the rule above:

- **A newline ends a claim, exactly as a comma does.** Segmentation hands over whole paragraphs
  and this model writes numbered lists inside them, so a list of eight computations arrived as a
  single chunk whose leftmost operand was a symbol, and all eight were discarded together.
- **A chain of equalities is judged pairwise.** In `a_2 = a_1 + 1 = 0 + 1 = 1` the first terms are
  symbolic and the last two are a closed-form claim anyone can check. Demanding that *every* term
  be numeric threw it away; checking adjacent pairs reads the part that qualifies and skips the
  part that does not.

Three guards were added alongside, every one of them a false positive found in real output rather
than imagined: a leading list marker (`1.`) must not be absorbed into the first operand, where it
turns `1. 5 + 5 = 10` into the false `1.5 + 5 = 10`; a value that is not finite (`4/0`) is an
undefined quantity rather than a false claim, because `zoo - zoo` compares unequal to zero; and a
**base subscript is not a digit separator**, because Python reads `11111111_2` as the integer
111111112 and `58_9` as 589, which turns a true base conversion into a false claim. That last one
was found by a specimen search over 7,358 chains, where 23 of the 51 candidates it returned were
all this single bug wearing different problems.

The preflight below tests all six, alongside the page's own canonical example, cancelling the
sixes in 16/64, which the checker gets **wrong in the generous direction**: the arithmetic is true
and the method is nonsense. That is the limit a deterministic checker cannot cross, and it is why
section 6 brings in a model.


In [ ]:
from sympy import simplify, sympify


def _to_sympy_text(s):
    """LaTeX-ish maths to something sympify can read. Deliberately narrow."""
    s = re.sub(r"\\left|\\right", "", s)
    s = re.sub(r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", s)
    s = (s.replace("\\times", "*").replace("\\cdot", "*").replace("\\div", "/")
          .replace("^", "**").replace("$", "").replace("\\", ""))
    return re.sub(r"\s+", "", s)


def numeric_value(text):
    """A sympy number if the text is closed-form numeric, else None.

    Any letter disqualifies it, which rejects 'x', 'f(2)', 'pi' and 'sqrt' alike. That
    costs recall and buys the only thing that matters here: no false positives.

    The finiteness test is not decoration. A real step here contained '4/(2^2 - 4)',
    which sympifies to complex infinity, and `zoo - zoo` is `nan`, which compares
    unequal to zero and was duly reported as a false claim. It is not one. It is an
    undefined value, and a reader asked to check that label by hand would have nothing
    to check.
    """
    t = _to_sympy_text(text)
    if not t or not re.search(r"\d", t) or re.search(r"[A-Za-z]", t):
        return None
    # A digit, an underscore, a digit is a BASE SUBSCRIPT in this corpus, and Python
    # reads an underscore inside a numeral as a digit separator. So sympify turns
    # '11111111_2' into 111111112 and '58_9' into 589, and a TRUE base-conversion
    # step is published as a false claim. This was not hypothetical: a specimen
    # search over 7,358 easy chains returned 23 candidates that were all this bug,
    # and they outranked every real one because they looked short and clean.
    # Refuse rather than guess at the base. Judging '11111111_2 = 255' correctly
    # would mean implementing base conversion, and the one thing this checker has
    # is that a reader can verify its labels by hand.
    if re.search(r"\d_\d", t):
        return None
    try:
        e = sympify(t, rational=True)
    except Exception:
        return None
    if not getattr(e, "is_number", False) or e.free_symbols:
        return None
    return e if e.is_finite is True else None


def _strip_prose(side):
    """Drop whole words of prose from the ends of one side of an equation.

    'So 8 * 7' -> '8 * 7' and '56 apples' -> '56': a claim does not stop being checkable
    for having a sentence wrapped round it. Four guards stop this inventing claims that
    were never made, and each one is a false positive that testing actually produced:

    * only tokens of TWO OR MORE letters go. Stripping the 'n' from 'n + 1' would leave
      '+ 1', and 'Let n = 4, so n + 1 = 5' would read as the false claim 1 = 5.
    * a leading list marker goes. This model writes numbered steps, so '1. 5 + 5 = 10'
      is common, and without this the marker joins the first operand: '1. 5 + 5' parses
      as '1.5+5' and a true claim reads as the false 6.5 = 10. Only a bare '1.' or '2)'
      matches, and only when tokens follow it, so the decimal in '2.5 + 2.5 = 5' and the
      sentence-ending '54.' in 'So 8 * 7 = 54.' both survive.
    * 'f(2)' is one token and is not purely alphabetic, so it survives, meets the
      no-letters rule and is skipped, rather than being mined for a '(2)'.
    * if stripping leaves the side starting on an operator we amputated an operand, so
      refuse to judge it at all.
    * if stripping leaves a BARE NUMBER, that number may have been part of the phrase
      rather than an operand, so refuse as well. This is the guard that matters most in
      practice: without it 'Sum of proper divisors of 284 = 1 + 2 + 4 + 71 = 78' reduces
      on the left to '284' and a correct step is published as the false 284 = 78.
    """
    toks = side.split()
    stripped = False
    while toks and (toks[0].endswith(":")                      # a step label, 'Step 3:'
                    or (len(toks) > 1                          # a list marker, '1.' or '2)'
                        and re.fullmatch(r"\d+[.)]", toks[0]))
                    or (toks[0].isalpha() and len(toks[0]) > 1)):
        toks.pop(0)
        stripped = True
    while toks and toks[-1].isalpha() and len(toks[-1]) > 1:
        toks.pop()
    out = " ".join(toks).rstrip(".")
    if stripped:
        if re.match(r"^[+*/^]|^-\s", out):       # we amputated an operand
            return ""
        # A bare number left behind by front-stripping may have been part of the
        # phrase rather than an operand. 'Sum of proper divisors of 284' reduces to
        # '284', and reading that as a claim turned the TRUE step
        # '...of 284 = 1 + 2 + 4 + 71 = 78' into the false 284 = 78. So after
        # stripping words, the remains have to look like an expression. Stripping
        # UNITS off the back is not affected: '56 apples' -> '56' is still the same
        # number, which is why only the front loop sets this flag.
        if not re.search(r"\d\s*(\*\*|[-+*/^])|\\(cdot|times|div|frac)", out):
            return ""
    return out


def check_step(text):
    """-> (label, judged_claim). A step is false if ANY numeric claim in it is.

    Two decisions about how a claim is found, and together they are the difference
    between a measurement and an empty table: on this model's chains they take the share
    of steps the checker can judge from 0.1% to about 7%.

    **A newline ends a claim, exactly as a comma does.** Segmentation hands whole
    paragraphs to this function and this model writes numbered lists inside them, so
    without splitting on newlines a list of eight computations arrives as one chunk,
    the leftmost operand is a symbol, and all eight are discarded together.

    A third decision is defensive rather than about recall: a markdown bullet is
    stripped from the front of a chunk. That one is not a nicety. Without it the '-' in
    '- $5! = 120$' parses as negation, sympy reads '5!' as a factorial because this
    version enables that notation, and a true claim is published as the false -120 = 120.

    **A chain of equalities is judged pairwise, not all-or-nothing.** In
    'a_2 = a_1 + 1 = 0 + 1 = 1' the first two terms are symbolic and the last two are a
    closed-form claim a reader can check by hand. Requiring every term to be numeric
    throws that claim away. Checking ADJACENT pairs keeps the precision rule exactly as
    stated, both sides closed-form numeric with no symbols, and reads the part of the
    chain that qualifies.
    """
    # Thousands separators first: splitting on commas below would cut '1,024' in half
    # and turn a valid claim into an unparseable one.
    text = re.sub(r"(?<=\d),(?=\d\d\d\b)", "", text)

    judged = None
    for chunk in re.split(r"[,;\n]|\bso\b|\bSo\b|\bthus\b|\band\b|\bTherefore\b", text):
        if "=" not in chunk:
            continue
        # A markdown bullet, removed here rather than in _strip_prose because this is
        # the only place it can appear: chunks are split on newlines, so a leading
        # "- " is a list marker at the start of a line. Left in, it reads as a unary
        # minus, and '- $5! = 120$' became the false claim -120 = 120. The trailing
        # whitespace in the pattern is what keeps it off real negation: this model
        # writes '-\frac{2}{3}' and '-5' with no space, so neither matches.
        chunk = re.sub(r"^\s*[-*+]\s+", "", chunk)
        parts = [numeric_value(_strip_prose(p)) for p in chunk.split("=")]
        for a, b in zip(parts, parts[1:]):
            if a is None or b is None:
                continue
            judged = chunk.strip()
            if simplify(a - b) != 0:
                return "false", judged
    return ("true", judged) if judged else ("unknown", None)


In [ ]:
PREFLIGHT = [
    # --- claims it should judge ------------------------------------------------
    ("3/4 = 9/12",                       "true"),
    ("17/12 - 5/12 = 1",                 "true"),
    ("2 + 2 = 5",                        "false"),
    ("So 8 * 7 = 54.",                   "false"),   # prose in front of the claim
    ("Step 3: 100 - 37 = 62",            "false"),   # a step label in front of it
    ("We get 8 * 7 = 56 apples",         "true"),    # units after it
    ("1,024 = 1024",                     "true"),    # a separator, not a chunk break
    (r"\frac{3}{4} = \frac{9}{12}",      "true"),
    (r"$5 \times 4 = 20$",               "true"),
    # a phrase ending in a number is not an operand: judged on its second pair
    ("Sum of proper divisors of 284 = 1 + 2 + 4 + 71 = 78", "true"),
    ("Sum of proper divisors of 284 = 1 + 2 + 4 + 71 = 79", "false"),
    ("The total is -5 = -5",             "unknown"),  # '-5' alone could be the phrase
    # --- the numbered lists this model actually writes -------------------------
    ("1. 5 + 5 = 10",                    "true"),    # '1.' is a marker, not an operand
    ("2) 8 * 7 = 56",                    "true"),
    ("1. 5 + 5 = 11",                    "false"),
    ("1. 2 + 2 = 4\n2. 3 + 3 = 7",       "false"),   # a newline ends a claim
    ("2.5 + 2.5 = 5",                    "true"),    # a decimal is not a marker
    ("6 * 7 = 42.",                      "true"),    # nor is a full stop
    ("- $5! = 120$",                     "true"),    # a bullet, not a minus sign
    ("* 3! = 6",                         "true"),
    ("- $5! = 121$",                     "false"),   # still judged, just not negated
    (r"-\frac{2}{3} = -\frac{4}{6}",     "true"),    # real negation, no space
    ("The answer is 12.",                "unknown"),
    # --- equality chains: judge the closed-form part, skip the symbolic part ----
    ("a_2 = a_1 + 1 = 0 + 1 = 1",        "true"),
    ("a_2 = a_1 + 1 = 0 + 1 = 2",        "false"),
    ("x = 2 * 3 = 7",                    "false"),
    # --- claims it must refuse: each is a false positive waiting to happen ------
    ("Let x = 5",                        "unknown"),  # a definition, not a claim
    ("f(2) = 7",                         "unknown"),  # depends on an earlier definition
    ("Let n = 4, so n + 1 = 5",          "unknown"),  # dropping 'n' would leave '+1 = 5'
    ("Substituting, 2x = 10",            "unknown"),  # an unknown in it
    ("x = 3 and y = 4",                  "unknown"),
    (r"\sqrt{16} = 4",                   "unknown"),  # true, but not closed-form here
    ("11111111_2 = 255",                 "unknown"),  # a base subscript. Python would
    ("58_9 - 18_9 = 40_9",               "unknown"),  # read these as 111111112 and 589
    ("(3, 5) = (3, 5)",                  "unknown"),  # a tuple, not arithmetic
    ("The common denominator is 12",     "unknown"),  # no claim at all
    ("The area is 12 square units",      "unknown"),
    ("4/0 = 4/0",                        "unknown"),  # undefined is not a false claim
    ("2/(2 - 2) = 1",                    "unknown"),
    # --- the blind spot, as a test rather than as a caveat ---------------------
    ("Cancel the 6s, so 16/64 = 1/4",    "true"),     # arithmetic true, method nonsense
]

fails = 0
for text, expect in PREFLIGHT:
    got, claim = check_step(text)
    if got != expect:
        fails += 1
        print(f"FAIL {text!r:<38} -> {got:<8} expected {expect}")
assert not fails, f"{fails} of {len(PREFLIGHT)} checker preflight cases behaved unexpectedly"

print(f"checker preflight: {len(PREFLIGHT)}/{len(PREFLIGHT)} as expected")
print(f"  {sum(1 for _, e in PREFLIGHT if e != 'unknown')} it judges, "
      f"{sum(1 for _, e in PREFLIGHT if e == 'unknown')} it refuses")
print("\nThe last case is the one to notice: 'cancel the 6s, so 16/64 = 1/4' comes back TRUE.")
print("The arithmetic is right and the method is nonsense, and no deterministic checker can")
print("tell those apart. That gap is what section 6 is for.")


In [ ]:
def delimiter_of(text):
    """Which rule in `harness.step_spans`' cascade actually divided this chain.

    Recorded because it decides whether the PRM in section 5 is being asked to score
    at positions its own convention would produce. That checkpoint's released usage
    splits a response on BLANK LINES and marks each step end, so a chain we cut on
    single newlines or on sentence boundaries is scored somewhere else entirely. This
    mirrors the cascade rather than reaching into it: same patterns, same order, and
    it only asks which one first yields more than one piece.
    """
    text = harness.strip_empty_think(text)
    for name, pattern in (("paragraph", r"\n\s*\n"), ("line", r"\n"),
                          ("sentence", r"(?<=[.!?])\s+")):
        if len([p for p in re.split(pattern, text) if p.strip()]) > 1:
            return name
    return "whole chain"


for r in harness.progress(records, desc="checking steps", unit="chain"):
    steps = harness.segment_steps(r["trace"], r["tokens"], min_step_chars=MIN_STEP_CHARS)
    for s in steps:
        s["check"], s["claim"] = check_step(s["text"])
    r["steps"] = steps
    r["n_steps"] = len(steps)
    r["delimiter"] = delimiter_of(r["trace"])
    r["first_false"] = next((i for i, s in enumerate(steps) if s["check"] == "false"), None)
    r["n_false"] = sum(s["check"] == "false" for s in steps)
    r["n_checkable"] = sum(s["check"] in ("true", "false") for s in steps)

all_steps = [s for r in records for s in r["steps"]]
lab = Counter(s["check"] for s in all_steps)

harness.table(
    "step labels",
    ["label", "steps", "share"],
    [[k, lab[k], f"{lab[k]/max(len(all_steps),1):.1%}"] for k in ("true", "false", "unknown")],
    note=f"{len(all_steps):,} steps over {len(records)} chains; "
         f"median {median([r['n_steps'] for r in records]):.0f} steps per chain")

DELIMITER_MIX = Counter(r["delimiter"] for r in records)
harness.table(
    "what divided each chain into steps",
    ["rule", "chains", "share"],
    [[k, v, f"{v/max(len(records),1):.0%}"] for k, v in DELIMITER_MIX.most_common()],
    note="'paragraph' matches the PRM's own convention; anything else scores it "
         "off-convention and section 6 has to carry that")

print(f"\nchains with at least one step the checker could judge: "
      f"{sum(r['n_checkable'] > 0 for r in records)}/{len(records)}")
if lab["unknown"] / max(len(all_steps), 1) > 0.8:
    print("Most steps are beyond a deterministic checker, which is itself the argument for")
    print("section 6 and worth saying on the page rather than hiding.")
if DELIMITER_MIX["paragraph"] / max(len(records), 1) < 0.5 and USE_PRM:
    print("\nUnder half of these chains split on blank lines, so most of them are being")
    print("cut where the PRM's convention would not cut them. Expect its scores to be")
    print("worse than the checkpoint's reported quality, and read section 6 as measuring")
    print("THIS pairing rather than the PRM in general.")

In [ ]:
# ---------------------------------------------------------------------------
# Specimen fishing (approach A). Generate SHORT easy chains to draw the three worked
# examples from, and grade them with the same arithmetic checker. The generator is
# loaded and FREED here, before section 5 loads the PRM, so a 3B generator and a 7B
# PRM never sit on the card together. These chains are kept OUT of `records`: they
# never enter sections 7-9's measurements, they only source legible specimens.
# ---------------------------------------------------------------------------
fish_records = []
if FISH_SPECIMENS:
    easy = sorted((r for r in math500 if int(r["level"]) in FISH_LEVELS),
                  key=lambda r: r["unique_id"])
    random.Random(SEED).shuffle(easy)
    easy = easy[:FISH_N_PROBLEMS]
    print(f"fishing over {len(easy)} problems at levels "
          f"{'+'.join(map(str, FISH_LEVELS))}, {FISH_SAMPLES_PER_PROBLEM} samples each, "
          f"T={FISH_TEMPERATURE}, cap {FISH_CAP}")

    device = get_device()
    loaded = harness.load_model(PRIMARY, device=device, dtype=harness.pick_dtype(device))
    gmodel, gtok = loaded["model"], loaded["tokenizer"]
    geos = harness.end_of_turn_ids(gtok)
    fish_store = harness.TraceStore(OUT_DIR / "fish_traces.jsonl",
                                    run_id=f"fish-{ITEM_SET_HASH[:8]}",
                                    key_fields=("problem_id", "sample_index"))

    @torch.inference_mode()
    def sample_easy(question, k, seed):
        """k natural-ended chains for one easy problem. Sampled like page 7's pile, with
        this notebook's step-per-line suffix so the steps segment cleanly. No forced
        answer: a chain that hits the cap is dropped, not force-closed, because this page
        grades only chains the model ended itself."""
        chat = gtok.apply_chat_template(
            [{"role": "user", "content": render_prompt(question) + COT_SUFFIX}],
            tokenize=False, add_generation_prompt=True)
        prompt_ids = gtok(chat, add_special_tokens=False).input_ids
        inp = torch.tensor([prompt_ids], device=gmodel.device)
        torch.manual_seed(seed)
        out = gmodel.generate(
            input_ids=inp, attention_mask=torch.ones_like(inp), max_new_tokens=FISH_CAP,
            do_sample=True, temperature=FISH_TEMPERATURE, top_p=FISH_TOP_P, top_k=None,
            num_return_sequences=k,
            pad_token_id=gtok.pad_token_id if gtok.pad_token_id is not None else gtok.eos_token_id)
        chains = []
        for row in out:
            gen = row[inp.shape[1]:].tolist()
            stop = next((i for i, t in enumerate(gen) if t in geos), None)
            if stop is None:
                continue                    # hit the cap; not the model's own ending
            pieces = harness.incremental_token_strings(gen[:stop], gtok)
            chains.append({"trace": "".join(pieces), "tokens": pieces})
        return chains

    for row in harness.progress(easy, desc="fishing", unit="item"):
        seed = int(harness.sha256_of([str(SEED), row["unique_id"], "fish"])[:15], 16)
        for si, ch in enumerate(sample_easy(row["problem"], FISH_SAMPLES_PER_PROBLEM, seed)):
            extracted = extract_final_candidate(ch["trace"])
            fish_records.append(fish_store.append({
                "problem_id": row["unique_id"], "sample_index": si,
                "level": int(row["level"]), "subject": row["subject"],
                "question": row["problem"], "ground_truth": row["answer"],
                "trace": ch["trace"], "tokens": ch["tokens"],
                "truncated": False, "forced": False,
                "temperature": FISH_TEMPERATURE, "top_p": FISH_TOP_P,
                "chain_budget": FISH_CAP,
                "extracted_answer": extracted,
                "outcome_correct": bool(grade_answer(extracted, row["answer"])),
            }))

    del gmodel, gtok, loaded
    harness.free_memory()

    # Grade with the same checker (CPU), exactly as `records` were graded above.
    for r in harness.progress(fish_records, desc="checking fished chains", unit="chain"):
        steps = harness.segment_steps(r["trace"], r["tokens"], min_step_chars=MIN_STEP_CHARS)
        for s in steps:
            s["check"], s["claim"] = check_step(s["text"])
        r["steps"] = steps
        r["n_steps"] = len(steps)
        r["delimiter"] = delimiter_of(r["trace"])
        r["first_false"] = next((i for i, s in enumerate(steps) if s["check"] == "false"), None)
        r["n_false"] = sum(s["check"] == "false" for s in steps)
        r["n_checkable"] = sum(s["check"] in ("true", "false") for s in steps)

    n_right = sum(r["outcome_correct"] for r in fish_records)
    short_lucky = sorted(
        [r for r in fish_records if r["outcome_correct"] and r["n_false"] > 0
         and r["n_steps"] <= FISH_MAX_STEPS],
        key=lambda r: r["n_steps"])
    print(f"\n{len(fish_records)} natural-ended easy chains, {n_right} right")
    print(f"SHORT LUCKY candidates (<= {FISH_MAX_STEPS} steps, checker-proved false step, "
          f"right answer): {len(short_lucky)}")
    for r in short_lucky[:12]:
        print(f"  {r['problem_id']:<30} L{r['level']} {r['n_steps']:>2} steps  "
              f"break@{r['first_false']}  {r['question'][:52]}")
    if not short_lucky:
        print("  none found. Widen FISH_LEVELS toward 3, raise FISH_SAMPLES_PER_PROBLEM, or")
        print("  raise FISH_MAX_STEPS. The page falls back to the L4-5 pinned specimen.")
else:
    print("FISH_SPECIMENS off; specimens come from page 7's L4-5 pile (see section 10).")


## 5. The process reward model

The checker cannot see method errors. A model can, and `Qwen2.5-Math-PRM-7B` emits a score per
reasoning step, which is process supervision in the literal sense.

But a PRM's labels are opinions, and putting them on the page unexamined would repeat exactly the
defect this refactor exists to remove: the current page reveals hand-written booleans and calls
them a PRM's verdict. So the PRM does not get to be the authority here. It gets **measured**, in
section 6, against the checker on the steps both can see, because on its own subset the checker
is right by construction.

### What could be wrong with these labels, named before they are collected

**Not its competence.** The PRM is far better at mathematics than the 3B whose work it grades,
and that is the right direction for a verifier rather than a problem with one. A grader no
stronger than the thing it grades is the useless case.

**Its lineage, which is not independent.** It is the same Qwen2.5 pretraining lineage as the
generator, on the math-specialised branch. So it is a test of whether a *bigger, specialised*
checkpoint catches this one's errors, not of whether an *unrelated* grader does, and any error
mode the two inherited together is invisible to this pairing. Page 8's notebook carries the same
limitation for its 7B judge and says so; this page says so too.

**Distribution shift, which is the one to actually watch.** Its step labels were learned on
terse, LaTeX-dense rollouts from math-specialised models with steps delimited by blank lines.
This page feeds it a chatty 3B-Instruct chain segmented by whatever rule section 4 printed. The
failure that produces is not random noise, it is **low specificity**: sound-but-verbose steps
marked down for being unlike the training distribution. That would inflate the PRM's detection
row in section 7 and would make localisation meaningless if localisation depended on it, which
is why sections 7 and 8 are the checker's and the PRM only ever adds a second row.

So section 6 measures agreement, AUC and specificity, and section 2's gate decides whether the
PRM may appear on the page at all. If it is off, will not load, or fails either preflight,
everything below still runs from the checker alone and `derived.json` records that it did not
participate.

**And a fourth thing, which is not about the PRM but about how we load it.** This checkpoint
ships its own modeling code, and that code is written against an older transformers than the one
this project pins. Used as-is it fails twice loudly and then once *quietly*: the forward pass
returns finite hidden states that are simply wrong, so every step scores about 0.25 and a
known-false step separates from a true one by 0.002. Nothing raises, and the natural conclusion
would have been "this PRM cannot grade our chains". The next cell therefore loads the body with
transformers' own `Qwen2Model` and attaches the checkpoint's score head by hand, and preflights
the body by asking it for the capital of France before trusting anything it says about
mathematics. Same weights, same arithmetic, no vendored code.

**A bug in the next cell is not a missing checkpoint.** An earlier draft caught every exception
around the load, so a `NameError` in this notebook printed "PRM unavailable", fell through to
checker-only, and published a full set of numbers under a `FULL` label. Programming errors are
re-raised here; only genuine load failures fall back.

In [ ]:
# ---------------------------------------------------------------------------
# Adopting this page's own PRM scores, which is what makes REPLAY a replay.
# ---------------------------------------------------------------------------
# Every other page's REPLAY reads out/page_NN/traces.jsonl and re-derives its
# figures from it. This page's read page 7's pile and re-graded from scratch,
# which is a re-derivation rather than a replay, and it is why REPLAY could not
# produce the PRM half at all: page 7's chains carry no scores, and computing
# them needs 15 GB of weights and a 16 GB card. A reader could re-derive every
# checker number on this page and none of the PRM ones.
#
# So when this page's own traces are on disk AND were produced by the same PRM
# under the same conditions, adopt the scores and re-derive everything else. The
# checker still runs from scratch, every gate still runs, every aggregate is
# recomputed. The single thing taken on trust is the tensor arithmetic, and two
# independent FULL runs produced byte-identical traces before this was written,
# so adopting is equivalent to recomputing on this hardware rather than a hope.
#
# THE FINGERPRINT IS THE WHOLE SAFETY ARGUMENT. A score depends on the
# checkpoint, its revision, the quantisation, and where the steps were cut, so
# all four are in the key and a mismatch refuses rather than guesses. Anything
# looser and a stale cache reads as a fresh measurement, which is the failure
# mode section 5 already warns about: nothing raises, the numbers are wrong.

STEP_SEP = "<extra_0>"      # the token this PRM scores at, one per step
PRM_ADOPT_SCORES = True     # False forces a rescore even when a valid cache exists

PRM_CACHE, PRM_ADOPTED = {}, None
# The download ships flat, so both files may sit beside the notebook rather than
# under out/page_09/. Section 3 already accepts either for the traces; the scores
# and the manifest that vouches for them have to travel the same way, or a reader
# replaying the download gets the checker half and silently loses the PRM half.
_src_traces = next((q for q in (TRACES_PATH, Path("traces.jsonl")) if q.exists()), None)
_src_manifest = next(
    (q for q in (OUT_DIR / "manifest.json", Path("manifest.json")) if q.exists()), None)


def _prm_fingerprint(model_id, revision, quant, min_step_chars, sep):
    return "|".join([model_id, revision or "unpinned", quant or "none",
                     str(min_step_chars), sep])


if PRM_ADOPT_SCORES and _src_traces is not None and _src_manifest is not None:
    src = json.loads(_src_manifest.read_text(encoding="utf-8"))
    sp = (src.get("graders") or {}).get("prm") or {}
    seg = (src.get("step_segmentation") or {}).get("min_step_chars")
    want = _prm_fingerprint(PRM_MODEL, harness.repo_revision(PRM_MODEL) or sp.get("revision"),
                            PRM_QUANT, MIN_STEP_CHARS, STEP_SEP)
    have = _prm_fingerprint(sp.get("id", ""), sp.get("revision"), sp.get("quantisation"),
                            seg, STEP_SEP)
    if not sp:
        print("a previous run is on disk but recorded no PRM; nothing to adopt")
    elif want != have:
        # Refuse rather than mix. The two strings are printed because the whole
        # point of the key is that a reader can see WHY it was refused.
        print("PRM cache refused, the conditions differ:")
        print(f"  this run  {want}")
        print(f"  on disk   {have}")
    else:
        for line in _src_traces.open(encoding="utf-8"):
            if not line.strip():
                continue
            r = json.loads(line)
            sc = [s.get("prm") for s in r.get("steps", [])]
            if r.get("prm_ok") and all(isinstance(v, (int, float)) for v in sc) and sc:
                PRM_CACHE[(r["problem_id"], r.get("sample_index", 0))] = sc
        PRM_ADOPTED = {
            "run_id": src.get("run_id"), "run_mode": src.get("run_mode"),
            "notebook_sha256": src.get("notebook_sha256"),
            "timestamp_utc": src.get("timestamp_utc"),
            "fingerprint": want, "n_chains": len(PRM_CACHE),
            # The preflight needs the model, so it cannot be re-run here. Carried
            # forward and labelled, because dropping it silently would delete the
            # evidence that the loader was sane when these scores were made.
            "separation_preflight": sp.get("separation_preflight"),
            "quantisation": sp.get("quantisation"),
            "revision": sp.get("revision"),
        }
        print(f"adopting {len(PRM_CACHE):,} chains of PRM scores from run "
              f"{PRM_ADOPTED['run_id']} ({PRM_ADOPTED['run_mode']}, "
              f"{PRM_ADOPTED['timestamp_utc']})")
        print(f"  fingerprint {want}")

# Does the cache cover this run's population? A partial one is not usable: the
# reliability fit and the detection rate would be measured over a different set of
# chains than the checker's, and the page would compare two populations.
_keys = {(r["problem_id"], r.get("sample_index", 0)) for r in records}
PRM_FROM_CACHE = bool(PRM_CACHE) and _keys.issubset(PRM_CACHE.keys())
if PRM_CACHE and not PRM_FROM_CACHE:
    print(f"cache covers {len(_keys & PRM_CACHE.keys()):,} of {len(_keys):,} chains, "
          f"which is partial; rescoring instead")
    PRM_CACHE, PRM_ADOPTED, PRM_FROM_CACHE = {}, None, False

# The model is loaded only when something still needs computing. This is what
# takes REPLAY from "cannot produce the PRM half" to "reproduces it in seconds".
USE_PRM = USE_PRM and not PRM_FROM_CACHE
if PRM_FROM_CACHE:
    print("the PRM is not loaded: every chain's scores are adopted, and the checker,"
          "\nthe reliability fit and every aggregate below are recomputed from scratch.")


In [ ]:
PRM_READY, prm_body, prm_head, prm_tok = False, None, None, None
PRM_PREFLIGHT = None


def load_prm(model_id):
    """The PRM as a stock Qwen2 body plus the checkpoint's own 2-class head.

    **Why this does not use the checkpoint's bundled modeling code**, which is the
    obvious way to load it and is what an earlier version of this cell did. That code
    is written against an older transformers, and under the version pinned in
    `requirements.txt` it fails three ways in a row: `config.pad_token_id` no longer
    exists, `DynamicCache.from_legacy_cache` is gone, and, once both are worked around,
    the forward pass silently returns *finite but wrong* hidden states. Wrong is the
    dangerous word there. Nothing raises. The PRM simply scores every step at about
    0.25 and separates a known-false step from a true one by 0.002, which reads as "this
    PRM is useless on our chains" when it actually means "we computed it wrong".

    The body is an ordinary `Qwen2Model`, so transformers' own implementation loads and
    runs it correctly, and the head is three tensors in the checkpoint that we can wire
    up ourselves. Same weights, same arithmetic, no vendored code. The preflight below
    is what proves it, and it exists because that is how the bug was found.
    """
    import logging
    import warnings

    from safetensors import safe_open
    from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen2Model

    # bitsandbytes warns once per int8 matmul that it is casting bf16 to fp16. That is
    # 28 layers x 7 projections x 1,580 chains, and in a notebook every one of them is a
    # retained output event: the first FULL run wrote a 55 MB .ipynb and most of it was
    # this sentence. Suppressed rather than avoided, because changing the compute dtype
    # to dodge it would change the scores, and these are the scores section 6 measured.
    warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
    logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
    logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

    print(f"[1/3] fetch      {model_id}")
    path = Path(harness.fetch(model_id))
    tok = AutoTokenizer.from_pretrained(path)

    print(f"[2/3] load       body ({PRM_QUANT or 'bf16'})")
    if PRM_QUANT == "8bit":
        # The head is skipped: its output dimension is 2, and bitsandbytes' int8 matmul
        # fails outright on a matrix that narrow. It is 26 MB, so keeping it in full
        # precision costs nothing and removes the quantisation from the scores that
        # actually decide a label.
        body = Qwen2Model.from_pretrained(
            path, quantization_config=BitsAndBytesConfig(
                load_in_8bit=True, llm_int8_skip_modules=["score"])).eval()
    else:
        device = get_device()
        body = Qwen2Model.from_pretrained(
            path, dtype=harness.pick_dtype(device)).to(device).eval()
    dev = next(body.parameters()).device

    print("[3/3] attach     the checkpoint's score head")
    index = path / "model.safetensors.index.json"
    keys = ("score.0.weight", "score.0.bias", "score.2.weight", "score.2.bias")
    where = (json.loads(index.read_text())["weight_map"] if index.exists()
             else {k: "model.safetensors" for k in keys})
    w = {}
    for k in keys:
        with safe_open(path / where[k], framework="pt") as sf:
            w[k] = sf.get_tensor(k)
    h = body.config.hidden_size
    head = torch.nn.Sequential(torch.nn.Linear(h, h), torch.nn.ReLU(),
                               torch.nn.Linear(h, 2))
    head[0].weight.data, head[0].bias.data = w[keys[0]].float(), w[keys[1]].float()
    head[2].weight.data, head[2].bias.data = w[keys[2]].float(), w[keys[3]].float()
    head = head.to(dev).float().eval()
    print(f"      ready on {dev}")
    return body, head, tok


if USE_PRM:
    try:
        prm_body, prm_head, prm_tok = load_prm(PRM_MODEL)
        PRM_READY = True
    except (NameError, AttributeError, TypeError, SyntaxError, IndentationError):
        # A bug in THIS cell, not a missing checkpoint. Falling back here would publish
        # a checker-only run under a FULL label, which is the one failure this notebook
        # must not have. Let it break loudly.
        raise
    except ImportError as e:
        # Nor is a missing dependency a missing checkpoint, and this one has exactly
        # the shape the line above refuses: PRM_QUANT="8bit" needs bitsandbytes, an
        # environment without it arrives here, and falling through would publish a
        # checker-only run wearing a FULL label. requirements.txt carries it for this
        # reason, and an earlier version of that file had it commented out.
        raise RuntimeError(
            f"the PRM cannot load because a dependency is missing ({e}). Install it "
            f"with `pip install -r requirements.txt`, or set PRM_QUANT=None on a card "
            f"with room for 15 GB of bf16 weights, or run REPLAY. This is deliberately "
            f"not a fallback: FULL without the PRM is a checker-only run under a FULL "
            f"label.") from e
    except Exception as e:
        print(f"\nPRM unavailable ({type(e).__name__}: {e})")
        print("Continuing with the checker alone; sections 7 and 8 still produce their numbers.")
else:
    print(f"USE_PRM is off in {RUN_MODE}. The checker carries every measurement below.")


@torch.inference_mode()
def prm_scores(question, step_texts):
    """One score in [0,1] per step. Higher means the PRM thinks the step is sound."""
    body = STEP_SEP.join(t.strip() for t in step_texts) + STEP_SEP
    chat = prm_tok.apply_chat_template(
        [{"role": "system", "content": "Please reason step by step."},
         {"role": "user", "content": question},
         {"role": "assistant", "content": body}],
        tokenize=False, add_generation_prompt=False)
    enc = prm_tok(chat, return_tensors="pt").to(next(prm_body.parameters()).device)
    # use_cache=False: this is one scoring pass, never a generation, so the KV cache is
    # pure overhead.
    hidden = prm_body(**enc, use_cache=False).last_hidden_state
    probs = torch.softmax(prm_head(hidden.float()), dim=-1)
    sep_id = prm_tok.encode(STEP_SEP, add_special_tokens=False)[0]
    return probs[enc["input_ids"] == sep_id][:, 1].tolist()


@torch.inference_mode()
def prm_body_is_sane():
    """Does the transformer still know anything? The check that caught the real bug.

    A broken forward pass does not announce itself: it returns finite numbers of the
    right shape, and only the *content* is wrong. So ask it something a 7B maths model
    cannot get wrong and read the answer. The checkpoint ships an `lm_head` its reward
    class never uses, which is exactly the tool for this.
    """
    from safetensors import safe_open
    path = Path(harness.fetch(PRM_MODEL))
    index = path / "model.safetensors.index.json"
    where = (json.loads(index.read_text())["weight_map"]["lm_head.weight"]
             if index.exists() else "model.safetensors")
    with safe_open(path / where, framework="pt") as sf:
        w = sf.get_tensor("lm_head.weight")
    dev = next(prm_body.parameters()).device
    w = w.to(torch.float16).to(dev)
    enc = prm_tok("The capital of France is", return_tensors="pt").to(dev)
    hidden = prm_body(**enc, use_cache=False).last_hidden_state
    top = torch.topk((hidden[0, -1].to(torch.float16) @ w.T).float(), 5).indices.tolist()
    words = [prm_tok.decode([i]) for i in top]
    del w
    return any("paris" in x.strip().lower() for x in words), words


if PRM_READY:
    ok, words = prm_body_is_sane()
    print(f"\nbody sanity: 'The capital of France is' -> {words}")
    if not ok:
        PRM_READY = False
        print("The body does not know the capital of France, so its hidden states are")
        print("wrong however finite they look, and every score below would be noise")
        print("dressed as a measurement. Refusing the PRM.")

if PRM_READY:
    # A clean chain against the same chain with one step corrupted. A PRM that cannot
    # separate those is not a grader, and the margin has to be a real gap rather than
    # the right sign: a broken loader produced the right sign with a margin of 0.002.
    q = "What is 3/4 + 2/3?"
    clean = ["The common denominator is 12.", "3/4 = 9/12 and 2/3 = 8/12.",
             "9/12 + 8/12 = 17/12.", "So the answer is \\boxed{17/12}."]
    dirty = list(clean)
    dirty[2] = "9/12 + 8/12 = 15/12."                    # now arithmetically false

    a, b = prm_scores(q, clean), prm_scores(q, dirty)
    assert len(a) == len(clean), (
        f"PRM returned {len(a)} scores for {len(clean)} steps; the separator convention "
        f"for this checkpoint is not what this cell assumes")

    harness.table("PRM separation preflight", ["step", "clean", "corrupted"],
                  [[i + 1, f"{x:.3f}", f"{y:.3f}" + ("   <- corrupted" if i == 2 else "")]
                   for i, (x, y) in enumerate(zip(a, b))])

    PRM_PREFLIGHT = {"clean": a, "corrupted": b, "margin": a[2] - b[2],
                     "min_separation": PRM_MIN_SEPARATION,
                     "separates": bool(a[2] - b[2] >= PRM_MIN_SEPARATION)}
    print(f"\nseparation margin on the corrupted step: {PRM_PREFLIGHT['margin']:+.3f} "
          f"(needs {PRM_MIN_SEPARATION:+.2f})")
    print("Note what this preflight is and is not. These four steps are terse and")
    print("blank-line clean, which is the PRM's home distribution, so passing here says")
    print("the plumbing works. It does not say the scores transfer to the chains in")
    print("section 4, and that is what section 6 is for.")

    if not PRM_PREFLIGHT["separates"]:
        PRM_READY = False
        print("\nIt does not mark a known-false step meaningfully lower than a true one,")
        print("so its scores are not usable as labels. Falling back to the checker alone.")


In [ ]:
PRM_FAILURES = Counter()

if PRM_FROM_CACHE:
    # Adopted, not recomputed. Every score here was produced by the run named in
    # PRM_ADOPTED under a fingerprint this run matched; section 6 refits the
    # threshold on them from scratch and section 7 recounts every rate.
    for r in records:
        sc = PRM_CACHE.get((r["problem_id"], r.get("sample_index", 0)), [])
        r["prm_ok"] = len(sc) == len(r["steps"]) and len(sc) > 0
        if not r["prm_ok"]:
            PRM_FAILURES["adopted_score_count_mismatch"] += 1
            continue
        for s, v in zip(r["steps"], sc):
            s["prm"] = float(v)
    PRM_PREFLIGHT = PRM_ADOPTED["separation_preflight"]
    n_ok = sum(r.get("prm_ok", False) for r in records)
    print(f"adopted PRM scores for {n_ok}/{len(records)} chains from "
          f"{PRM_ADOPTED['run_id']}")
    if PRM_FAILURES:
        print(f"  chains whose adopted scores did not fit their steps: {dict(PRM_FAILURES)}")
    print("  The separation preflight is carried forward from that run and labelled")
    print("  as adopted in the manifest: it needs the model, so it cannot re-run here.")
elif PRM_READY:
    for r in harness.progress(records, desc="PRM scoring", unit="chain"):
        try:
            sc = prm_scores(r["question"], [s["text"] for s in r["steps"]])
        except (NameError, AttributeError, TypeError):
            raise                       # our bug, not this chain's fault
        except Exception as e:
            PRM_FAILURES[type(e).__name__] += 1
            sc = []
        r["prm_ok"] = len(sc) == len(r["steps"])
        if not r["prm_ok"]:
            if sc:
                PRM_FAILURES["score_count_mismatch"] += 1
            continue
        for s, v in zip(r["steps"], sc):
            s["prm"] = float(v)         # raw score only; the threshold is fitted below
    n_ok = sum(r.get("prm_ok", False) for r in records)
    print(f"\nPRM scored {n_ok}/{len(records)} chains")
    if PRM_FAILURES:
        print(f"  chains it could not score: {dict(PRM_FAILURES)}")
    print("  Scores are stored raw. Nothing is called sound or unsound yet, because the")
    print("  cut-off is a fitted quantity and section 6 is where it gets fitted.")
else:
    for r in records:
        r["prm_ok"] = False


def apply_prm_labels(threshold):
    """Turn raw PRM scores into step labels at one cut-off, and roll them up per chain.

    Kept separate from scoring so the threshold can move without a second forward
    pass, and so that a reader can see that the labels are a function of a number this
    notebook chose rather than a verdict the model handed down.
    """
    for r in records:
        if not r.get("prm_ok"):
            continue
        for s in r["steps"]:
            s["prm_label"] = "false" if s["prm"] < threshold else "true"
        r["prm_first_false"] = next(
            (i for i, s in enumerate(r["steps"]) if s["prm_label"] == "false"), None)
        r["prm_n_false"] = sum(s["prm_label"] == "false" for s in r["steps"])

## 6. Is the PRM worth believing?

On steps the checker judged, how well do the PRM's scores separate true from false? Where the
two disagree the checker is right, by construction, so this is a measurement of the PRM and not
a negotiation between two opinions.

**AUC is the headline, and the threshold is fitted rather than assumed.** An earlier draft
called a step unsound below 0.5 and then reported accuracy at that cut-off. A probability
calibrated on one distribution is not calibrated on another, so 0.5 was measuring where this
checkpoint's scores happen to sit on our chains as much as it was measuring the checkpoint. AUC
asks the threshold-free question, *does it rank a false step below a true one*, which is what a
process grader is actually for and is what page 8's spec demands of every verifier. The
operating point is then chosen on the same labelled steps and **recorded as fitted**, because a
number we picked from the data is not a property of the model.

Three readings at that operating point, answering different questions. **Agreement** is the
summary. **Recall on false steps** is whether it catches breaks the checker proved were there.
**Specificity** is whether it cries wolf on steps that were fine, and it is the one that decides
publication: a grader that flags a fifth of sound steps would drown section 8's localisation in
false breaks, and marking down verbose prose it was never trained on is precisely how this
pairing fails.

**The deepest caveat, and it belongs on the page rather than only here.** The checker judges
only closed-form arithmetic. Those steps are terse and numeric, which is to say they are the
steps *most* like the PRM's training distribution. So everything measured here is an
**optimistic** estimate of how the PRM does on the prose steps where it is actually being
trusted, and section 4 has probably already reported that those prose steps are most of the
chain. The reliability is measured where the checker could reach, and that is the easy ground
for both graders.

Whatever comes out is attached to any PRM-only claim the page makes. If the gate in section 2
fails, the PRM's rows are suppressed rather than footnoted.

In [ ]:
RELIABILITY, PRM_PUBLISHABLE = None, False
PRM_THRESHOLD = PRM_THRESHOLD_START

pairs = [(s["check"], s["prm"]) for r in records if r.get("prm_ok")
         for s in r["steps"] if s["check"] in ("true", "false") and "prm" in s]


def auc_of(ps):
    """P(a true step outranks a false one), ties at half credit.

    The Mann-Whitney reading of AUC, written out rather than imported because it is
    three lines and the whole point of this section is that a reader can check it.
    None when the checker labelled only one class, which is not a score of 0.5: it is
    no measurement at all, and the gate below treats it that way.
    """
    t = [v for lab, v in ps if lab == "true"]
    f = [v for lab, v in ps if lab == "false"]
    if not t or not f:
        return None
    wins = sum((x > y) + 0.5 * (x == y) for x in t for y in f)
    return wins / (len(t) * len(f))


def readings_at(ps, threshold):
    """Agreement, recall on false steps and specificity on true ones, at one cut-off."""
    got = [(lab, "false" if v < threshold else "true") for lab, v in ps]
    on_false = [g for lab, g in got if lab == "false"]
    on_true = [g for lab, g in got if lab == "true"]
    return {
        "threshold": threshold,
        "agreement": mean([lab == g for lab, g in got]),
        "recall_on_false": mean([g == "false" for g in on_false]) if on_false else None,
        "specificity_on_true": mean([g == "true" for g in on_true]) if on_true else None,
    }


if pairs:
    AUC = auc_of(pairs)
    at_start = readings_at(pairs, PRM_THRESHOLD_START)

    PLATEAU = None
    if PRM_FIT_THRESHOLD and AUC is not None:
        # Youden's J: the cut-off that best trades catching false steps against not
        # flagging true ones. Fitted on the checker's labels, which is the only ground
        # truth here, and recorded as fitted so nobody reads it as the model's own.
        vals = sorted({v for _, v in pairs})
        if len(vals) > 200:                       # keep the search cheap on a big pile
            vals = [vals[int(i * len(vals) / 200)] for i in range(200)]
        cands = [0.0] + [(a + b) / 2 for a, b in zip(vals, vals[1:])] + [1.0]

        def youden(t):
            rd = readings_at(pairs, t)
            if rd["recall_on_false"] is None or rd["specificity_on_true"] is None:
                return -1.0
            return rd["recall_on_false"] + rd["specificity_on_true"] - 1.0

        # The PLATEAU, not the argmax, and this is a fix rather than a refinement.
        # J is nearly flat up here, so `max()` returns whichever cut-off happens to
        # win a comparison between values differing by less than one step's worth of
        # J. Two runs of the SAME 1,580 chains through the same weights, agreeing on
        # AUC to five decimals, chose 0.747 and 0.804 and moved the PRM's detection
        # headline by 1.7 points. A number the page prints may not depend on that.
        #
        # So every cut-off within PRM_J_TOLERANCE of the best counts as TIED, and the
        # tie is broken by the measure that decides publication: specificity. Among
        # cut-offs that trade equally well, take the one that cries wolf least, which
        # is the lowest of them, since a lower cut-off calls fewer sound steps
        # unsound. Measured over the two runs that disagreed, this picks 0.747 and
        # 0.755, a spread of 0.008 rather than 0.057, and holds detection at 19.5%
        # and 19.8% rather than 19.5% and 21.3%.
        js = [(t, youden(t)) for t in cands]
        best_j = max(j for _, j in js)
        PLATEAU = [t for t, j in js if j >= best_j - PRM_J_TOLERANCE]
        PRM_THRESHOLD = min(PLATEAU)

    fitted = readings_at(pairs, PRM_THRESHOLD)
    apply_prm_labels(PRM_THRESHOLD)

    PRM_PUBLISHABLE = bool(
        len(pairs) >= PRM_MIN_COMPARED_STEPS
        and AUC is not None and AUC >= PRM_MIN_AUC
        and fitted["specificity_on_true"] is not None
        and fitted["specificity_on_true"] >= PRM_MIN_SPECIFICITY)

    n_false_steps = sum(1 for lab, _ in pairs if lab == "false")
    RELIABILITY = {
        "n_steps_compared": len(pairs),
        "n_false_steps": n_false_steps,
        "auc": AUC,
        "threshold": PRM_THRESHOLD,
        "threshold_source": (f"fitted on the checker's labels: of the cut-offs within "
                             f"{PRM_J_TOLERANCE} of the best Youden's J, the one with "
                             f"the highest specificity")
                            if PRM_FIT_THRESHOLD else "fixed",
        "threshold_plateau": ([min(PLATEAU), max(PLATEAU), len(PLATEAU)]
                              if PLATEAU else None),
        "j_tolerance": PRM_J_TOLERANCE if PRM_FIT_THRESHOLD else None,
        "threshold_start": PRM_THRESHOLD_START,
        "agreement": fitted["agreement"],
        "recall_on_false": fitted["recall_on_false"],
        "specificity_on_true": fitted["specificity_on_true"],
        "at_start_threshold": at_start,
        "gate": {"min_auc": PRM_MIN_AUC, "min_specificity": PRM_MIN_SPECIFICITY,
                 "min_compared_steps": PRM_MIN_COMPARED_STEPS, "passed": PRM_PUBLISHABLE},
        "measured_on": "steps with a closed-form numeric claim, which are the steps most "
                       "like the PRM's training distribution; an optimistic estimate of "
                       "its reliability on the prose steps it is trusted with",
    }

    def _pct(x):
        return f"{x:.1%}" if x is not None else "n/a"

    harness.table(
        "PRM against the arithmetic checker",
        ["measure", "value", "over"],
        [["AUC (threshold-free)", f"{AUC:.3f}" if AUC is not None else "n/a",
          f"{len(pairs)} steps"],
         ["operating point", f"{PRM_THRESHOLD:.3f}", RELIABILITY["threshold_source"]],
         ["agreement", _pct(fitted["agreement"]), f"{len(pairs)} steps"],
         ["recall on false steps", _pct(fitted["recall_on_false"]),
          f"{n_false_steps} steps"],
         ["specificity on true steps", _pct(fitted["specificity_on_true"]),
          f"{len(pairs) - n_false_steps} steps"]],
        note="where they disagree the checker is right, by construction")

    print(f"\nat the unfitted 0.5 cut-off, for contrast: agreement "
          f"{_pct(at_start['agreement'])}, recall {_pct(at_start['recall_on_false'])}, "
          f"specificity {_pct(at_start['specificity_on_true'])}")

    if PRM_PUBLISHABLE:
        print(f"\nGATE PASSED. The PRM's rows may appear, carrying AUC {AUC:.2f} and the")
        print("caveat that this was measured on the arithmetic steps rather than the prose.")
    else:
        why = []
        if len(pairs) < PRM_MIN_COMPARED_STEPS:
            why.append(f"only {len(pairs)} comparable steps (need {PRM_MIN_COMPARED_STEPS})")
        if AUC is None:
            why.append("AUC undefined: the checker labelled only one class of step, so "
                       "there is nothing to rank against")
        elif AUC < PRM_MIN_AUC:
            why.append(f"AUC {AUC:.2f} below {PRM_MIN_AUC:.2f}")
        if (fitted["specificity_on_true"] is not None
                and fitted["specificity_on_true"] < PRM_MIN_SPECIFICITY):
            why.append(f"specificity {_pct(fitted['specificity_on_true'])} below "
                       f"{PRM_MIN_SPECIFICITY:.0%}")
        print(f"\nGATE FAILED: {'; '.join(why)}.")
        print("The PRM's rows are suppressed below. This is a finding about grading this")
        print("generator's chains with this checkpoint, and it is worth one sentence on the")
        print("page: an off-the-shelf PRM pointed at off-distribution prose is not free.")
elif PRM_READY:
    print("no overlapping steps to compare on, so the PRM's reliability is unmeasured and")
    print("the page may not report PRM-only judgements")
else:
    print("no PRM in this run; every measurement below is the checker's")

In [ ]:
# PRM-score the fished chains at the threshold fitted on the L4-5 pile, so a specimen's
# opinion chips come from the same grader the page measured. The generator was freed in
# the fishing cell, so only the PRM is resident here.
if fish_records:
    if PRM_READY:
        for r in harness.progress(fish_records, desc="PRM scoring fished chains", unit="chain"):
            try:
                sc = prm_scores(r["question"], [s["text"] for s in r["steps"]])
            except (NameError, AttributeError, TypeError):
                raise
            except Exception:
                sc = []
            r["prm_ok"] = len(sc) == len(r["steps"])
            if not r["prm_ok"]:
                continue
            for s, v in zip(r["steps"], sc):
                s["prm"] = float(v)
            if PRM_PUBLISHABLE:
                for s in r["steps"]:
                    s["prm_label"] = "false" if s["prm"] < PRM_THRESHOLD else "true"
                r["prm_first_false"] = next(
                    (i for i, s in enumerate(r["steps"]) if s["prm_label"] == "false"), None)
                r["prm_n_false"] = sum(s["prm_label"] == "false" for s in r["steps"])
        print(f"PRM scored {sum(r.get('prm_ok', False) for r in fish_records)}/"
              f"{len(fish_records)} fished chains at the fitted {PRM_THRESHOLD:.3f}")
    else:
        for r in fish_records:
            r["prm_ok"] = False

    # ---- what the fishing pass is actually for ---------------------------
    # A shorter lucky chain is only an upgrade if it KEEPS the property the page's
    # third beat rests on: the checker proves the step false and the PRM calls it
    # sound. A candidate the PRM catches is a worse specimen than the incumbent
    # however legible it is, because the page loses its closing contrast. So the
    # PRM score at the break is a column here rather than something to look up
    # later. Nothing below installs anything; section 10 still prefers the pile.
    def _partners(pid):
        return sum(1 for r in fish_records if r["problem_id"] == pid
                   and not r["outcome_correct"] and r["n_false"] == 1
                   and r["n_steps"] > 1)

    cands = sorted([r for r in fish_records if r["outcome_correct"] and r["n_false"] > 0],
                   key=lambda r: (r["n_steps"], r["n_false"]))
    rows = []
    for r in cands[:25]:
        v = r["steps"][r["first_false"]].get("prm")
        rows.append([r["problem_id"].replace("test/", "").replace(".json", ""),
                     r["n_steps"], r["n_false"], r["first_false"] + 1,
                     _partners(r["problem_id"]),
                     f"{v:.3f}" if v is not None else "n/a",
                     ("MISSES it" if v >= PRM_THRESHOLD else "catches it")
                     if v is not None else "-"])
    harness.table(
        f"lucky candidates: {len(cands)} in {len(fish_records):,} fished chains",
        ["problem", "steps", "false", "break@", "partners", "PRM at break", "verdict"],
        rows,
        note="incumbent: number_theory/357 at 13 steps, break@4, PRM 0.970 (misses it), "
             "15 partners. A candidate has to beat that on legibility WITHOUT losing "
             "the miss")

    for r in cands[:6]:
        s = r["steps"][r["first_false"]]
        v = s.get("prm")
        print(f"\n{r['problem_id']}  L{r['level']}  {r['n_steps']} steps  "
              f"PRM {v:.3f}" if v is not None else
              f"\n{r['problem_id']}  L{r['level']}  {r['n_steps']} steps")
        print(f"  Q:     {r['question'][:86]}")
        print(f"  claim: {s['claim']}")


## 7. Detection: how often does outcome grading reward nonsense

Spec §3a. Among chains that reached the **right** answer, the share containing a step the checker
proved false. Outcome grading passes every one.

**This may come out small, and the spec says so in advance.** A false step usually propagates
into a wrong answer, and the cases where it does not are often method errors with valid
arithmetic, which the checker cannot see. If it is small, the page reports it as small and
changes its verb rather than inflating it.

The rate is over **rationales**, not problems: it is the contamination rate of page 12's STaR
filter (spec §1), which is why section 3 grades every chain rather than one per problem. So the
cell reports both the chain rate and the number of distinct problems behind it, because sixteen
chains from one problem are not sixteen independent observations.


In [ ]:
right = [r for r in records if r["outcome_correct"]]
wrong = [r for r in records if not r["outcome_correct"]]
right_bad = [r for r in right if r["n_false"] > 0]

# The chain-level rate is what the STaR framing asks for. The problem-level count is
# what says how much of it rests on how few problems, and both belong on the page:
# sixteen chains from one problem are not sixteen independent observations.
probs_right = {r["problem_id"] for r in right}
probs_bad = {r["problem_id"] for r in right_bad}

# The denominator the base rate does NOT have. `n_right` counts every chain that
# reached the right answer, including the ones where the checker had no closed-form
# numeric claim to read and was therefore silent BY CONSTRUCTION rather than because
# the chain was sound. Dividing by all of them reports the checker's blind spot as
# though it were evidence of clean reasoning. Both denominators go to the page: the
# unconditional rate for comparability with the PRM, which does score every chain,
# and the conditional one for what the checker actually looked at.
right_reachable = [r for r in right if r["n_checkable"] > 0]

DETECTION = {
    "grader": "arithmetic checker", "source": source,
    "n_right": len(right), "n_right_with_false_step": len(right_bad),
    "base_rate": len(right_bad) / max(len(right), 1),
    "n_right_reachable": len(right_reachable),
    "reachable_rate": len(right_bad) / max(len(right_reachable), 1),
    "n_wrong": len(wrong),
    "n_problems_with_a_right_chain": len(probs_right),
    "n_problems_with_a_contaminated_right_chain": len(probs_bad),
    "problem_rate": len(probs_bad) / max(len(probs_right), 1),
    "unit_of_independence": "the problem; the chain rate is clustered within problems",
    "prm_reported": False,
}

rows = [["outcome grader", f"{len(right):,} of {len(records):,} chains pass", ""],
        ["of those, the checker could read at all", len(right_reachable),
         f"{len(right_reachable) / max(len(right), 1):.1%}"],
        ["of those, a step the checker proved false", len(right_bad),
         f"{DETECTION['base_rate']:.2%}"],
        ["  same, over only the chains it could read", len(right_bad),
         f"{DETECTION['reachable_rate']:.2%}"]]

if PRM_PUBLISHABLE:
    pr = [r for r in right if r.get("prm_ok")]
    pr_bad = [r for r in pr if r.get("prm_n_false", 0) > 0]
    DETECTION["prm_reported"] = True
    DETECTION["prm_base_rate"] = len(pr_bad) / max(len(pr), 1)
    DETECTION["prm_n_chains"] = len(pr)
    DETECTION["prm_auc"] = RELIABILITY["auc"]
    DETECTION["prm_threshold"] = RELIABILITY["threshold"]
    rows.append(["of those, a step the PRM calls unsound", len(pr_bad),
                 f"{DETECTION['prm_base_rate']:.2%}"])
    note = (f"the PRM row also sees non-arithmetic steps; attach its measured "
            f"AUC {RELIABILITY['auc']:.2f} to it, and the fact that the AUC was "
            f"measured on arithmetic steps only")
elif RELIABILITY:
    note = ("the PRM ran and did not clear section 2's gate, so its row is suppressed "
            "rather than footnoted; the checker sees arithmetic only, so this is a "
            "floor rather than a rate")
    print("The PRM scored these chains and its row is withheld. The suppression is the")
    print("finding, not a gap: derived.json carries what it scored and why it failed.\n")
else:
    note = "the checker sees arithmetic only, so this is a floor rather than a rate"

harness.table("detection", ["", "count", "share of right answers"], rows, note=note)

n_zero = sum(1 for r in records if r["n_checkable"] == 0)
DETECTION["n_chains_with_no_checkable_step"] = n_zero
DETECTION["share_chains_with_no_checkable_step"] = n_zero / max(len(records), 1)
print(f"\n{n_zero:,} of {len(records):,} chains ({n_zero / max(len(records), 1):.1%}) carry no "
      f"closed-form numeric claim at\nall, so the checker is silent on them by construction. "
      f"The median chain has nothing\nto grade, and any rate taken over all chains is diluted "
      f"by that. Report it alongside.\n")

print(f"\nthose {len(right_bad)} chains come from {len(probs_bad)} distinct problems, "
      f"out of {len(probs_right)} that")
print(f"produced a right answer at all. Quote the chain rate, and quote that too: a rate")
print("clustered in a handful of problems is a different claim from one spread evenly.")

if DETECTION["base_rate"] == 0:
    print("\nZERO -- a fact about the checker before it is a fact about the model. See the")
    print("spec's open question 4 before reporting this as a finding about reasoning.")
elif DETECTION["base_rate"] < 0.05:
    print("\nSmall, exactly as the spec predicted. Report it as small and change the verb;")
    print("the STaR framing is what gives it stakes, because every one of these is a piece")
    print("of bad reasoning an outcome check would wave into the training set.")


## 8. Localisation: how much of a wrong chain was sound

Spec §3b, and the measurement Solution B on the current page has always taught without evidence.

Take chains that reached the **wrong** answer where the checker finds a break at step *k* of *n*.
Outcome grading can only say *wrong*, which condemns all *n*. Process grading points at *k* and
leaves the first *k-1* standing. So the quantity is `k/n`, the share of the chain that survives.

The second reading tests something the page asserts rather than shows: whether a break really is
one step, or whether real chains carry several.


In [ ]:
localisable = [r for r in wrong if r["first_false"] is not None and r["n_steps"] > 1]
LOCALISATION = {"n_wrong": len(wrong), "n_localisable": len(localisable),
                "n_problems": len({r["problem_id"] for r in localisable})}

if localisable:
    shares = sorted(r["first_false"] / r["n_steps"] for r in localisable)
    med = shares[len(shares) // 2]
    dist = Counter(r["n_false"] for r in localisable)
    LOCALISATION.update({
        "median_share_sound": med,
        "mean_share_sound": mean(shares),
        "share_with_one_false_step": dist[1] / len(localisable),
        "n_false_distribution": dict(sorted(dist.items())),
    })

    harness.table(
        "what each grader condemns, on a wrong chain with a findable break",
        ["grader", "verdict", "share of the chain condemned"],
        [["outcome", "wrong", "100%"],
         ["process", f"broken at step {med:.0%} of the way in", f"{1 - med:.0%}"]],
        note=f"{len(localisable)} of {len(wrong)} wrong chains have a break the checker can "
             f"find, over {LOCALISATION['n_problems']} distinct problems")

    print(f"\n-> the first {med:.0%} of the chain was sound, and outcome grading threw it away")
    print(f"\nbreaks per chain: {LOCALISATION['n_false_distribution']}")
    print(f"exactly one broken step: {LOCALISATION['share_with_one_false_step']:.0%}")
    if LOCALISATION["share_with_one_false_step"] < 0.5:
        print("  ^ under half, so the page must NOT claim breaks localise to a single line.")
        print("    Follow the data: say how many, and keep Solution B as one case, not the rule.")
    else:
        print("  ^ a majority, so Solution B's one-line break is the common case and the page")
        print("    may teach it as such. It is not the only case, and the distribution above is")
        print("    the honest version of 'usually one'.")
else:
    print(f"No wrong chain has a findable break, out of {len(wrong)} wrong chains.")
    print("Section 8 has nothing to plot. Either the checker's recall is too low on this")
    print("model's output, or these answers fail for non-arithmetic reasons.")


In [ ]:
import matplotlib.pyplot as plt

if localisable:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2),
                                   gridspec_kw={"width_ratios": [3, 2]})
    ax1.hist([s * 100 for s in shares], bins=10, range=(0, 100), color="#3b4a6b")
    ax1.axvline(med * 100, color="#c2703d", lw=2)
    ax1.set_xlabel("share of the chain before the break (%)")
    ax1.set_ylabel("wrong chains")
    ax1.set_title("where the break lands", fontsize=10, loc="left")

    ax2.barh([1, 0], [100, (1 - med) * 100], height=0.5, color=["#8c8c8c", "#3b4a6b"])
    ax2.set_yticks([1, 0]); ax2.set_yticklabels(["outcome", "process"])
    ax2.set_xlim(0, 100); ax2.set_xlabel("% of the chain condemned")
    ax2.set_title("what each grader throws away", fontsize=10, loc="left")

    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{PRIMARY} · MATH-500 L4-5 · n={len(localisable)} wrong chains with a "
                 f"findable break", fontsize=9, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_localisation.png", dpi=160, bbox_inches="tight")
    plt.show()


## 8b. Do the numbers survive a different idea of where a step ends?

Sections 7 and 8 rest on a segmentation rule, and a rule is a choice. `harness.step_spans` runs a
cascade, blank lines then single lines then sentences, and takes the first that divides the
chain, so different chains in this run were cut by different rules. Section 4 printed which.

This is not a footnote-sized worry. **A step is the unit both graders judge**, so cutting finer
produces more steps and more chances for the checker to find a false one, while cutting coarser
buries a false claim inside a step that also contains true ones. If the headline moves a lot
when the rule changes, then it is partly a measurement of our formatting convention and the page
has to say so.

Cheap to answer: force each rule on every chain and recompute. No inference, no model, no
generation, and it runs in `REPLAY`. The canonical row is what sections 7 and 8 reported; the
other three are what the same chains would have said under one rule throughout.

It also settles the PRM question from section 5 in the honest direction. The `paragraph` row is
the PRM's own convention, so the distance between it and the canonical row is roughly what the
PRM is being asked to absorb before it says anything at all.

In [ ]:
def resegment(text, pattern, min_step_chars=MIN_STEP_CHARS):
    """Step texts under one forced delimiter, with the cascade's merge rule for short
    fragments. Text only: the checker needs no token spans, and section 10's reveal
    keeps the canonical segmentation. This exists to answer one question and no other.
    """
    text = harness.strip_empty_think(text)
    pieces = [p.strip() for p in re.split(pattern, text) if p.strip()]
    merged = []
    for k, p in enumerate(pieces):
        short = (len(p) < min_step_chars and not re.search(r"\d", p)
                 and k != len(pieces) - 1 and "\\boxed" not in p)
        if merged and short:
            merged[-1] = merged[-1] + " " + p
        else:
            merged.append(p)
    return merged


def measure_under(recs, step_lists):
    """Detection and localisation from one segmentation of every chain."""
    rows = []
    for r, steps in zip(recs, step_lists):
        labels = [check_step(t)[0] for t in steps]
        rows.append(dict(
            correct=r["outcome_correct"], n=len(steps),
            n_false=sum(l == "false" for l in labels),
            first_false=next((i for i, l in enumerate(labels) if l == "false"), None)))
    rt = [x for x in rows if x["correct"]]
    wr = [x for x in rows if not x["correct"]]
    loc = [x["first_false"] / x["n"] for x in wr
           if x["first_false"] is not None and x["n"] > 1]
    return dict(mean_steps=mean([x["n"] for x in rows]) if rows else 0.0,
                detection=len([x for x in rt if x["n_false"] > 0]) / max(len(rt), 1),
                n_localisable=len(loc),
                median_share_sound=median(loc) if loc else None)


# Three more passes over every chain would cost more than the rest of the notebook, and
# a sensitivity check does not need the full n: it needs enough to show whether the
# answer moves. Seeded so the subsample is the same on every run, and the size is
# printed rather than buried.
SEG_SAMPLE = 400
seg_recs = (records if len(records) <= SEG_SAMPLE else
            random.Random(SEED).sample(records, SEG_SAMPLE))
print(f"segmentation sensitivity on {len(seg_recs):,} of {len(records):,} chains "
      f"(seeded subsample)\n")

SEGMENTATION_CHECK = {"n_chains": len(seg_recs),
                      "canonical": measure_under(seg_recs, [[s["text"] for s in r["steps"]]
                                                            for r in seg_recs])}
for name, pattern in (("paragraph", r"\n\s*\n"), ("line", r"\n"),
                      ("sentence", r"(?<=[.!?])\s+")):
    SEGMENTATION_CHECK[name] = measure_under(
        seg_recs, [resegment(r["trace"], pattern) for r in seg_recs])

harness.table(
    "the same chains, cut four ways",
    ["segmentation", "steps/chain", "detection", "wrong chains with a break", "share sound"],
    [[k, f"{v['mean_steps']:.1f}", f"{v['detection']:.2%}", v["n_localisable"],
      f"{v['median_share_sound']:.0%}" if v["median_share_sound"] is not None else "n/a"]
     for k, v in SEGMENTATION_CHECK.items() if isinstance(v, dict)],
    note="'canonical' is the cascade sections 7 and 8 actually used; the rest force one "
         "rule on every chain")

# Both headlines get judged, not just the first. Detection asks only whether a false
# claim is anywhere in the chain, and check_step splits on newlines itself, so it is
# largely immune to where the STEP boundaries fall. Localisation is the opposite: it is
# a ratio of step positions, so moving the boundaries moves it directly.
runs = [v for v in SEGMENTATION_CHECK.values() if isinstance(v, dict)]
det = [v["detection"] for v in runs]
snd = [v["median_share_sound"] for v in runs if v["median_share_sound"] is not None]
SEGMENTATION_CHECK["detection_spread_pts"] = (max(det) - min(det)) * 100
SEGMENTATION_CHECK["share_sound_spread_pts"] = (max(snd) - min(snd)) * 100 if snd else None

print(f"\ndetection ranges over {SEGMENTATION_CHECK['detection_spread_pts']:.2f} points")
if snd:
    print(f"share-sound ranges over {SEGMENTATION_CHECK['share_sound_spread_pts']:.0f} points "
          f"({min(snd):.0%} to {max(snd):.0%})")

if SEGMENTATION_CHECK["detection_spread_pts"] > 5:
    print("\nDETECTION moves with the rule, so section 7's number is partly a measurement")
    print("of our formatting convention. Report it with this range beside it.")
else:
    print("\nDetection barely moves, which is what we would expect: check_step splits on")
    print("newlines itself, so whether a false claim is FOUND hardly depends on where the")
    print("step boundaries were drawn. Section 7 may quote its number plainly.")

if snd and SEGMENTATION_CHECK["share_sound_spread_pts"] > 10:
    print("\nLOCALISATION does move, and by more than detection. That is not a surprise")
    print("either: 'how far into the chain' is a ratio of step positions, so redrawing the")
    print("boundaries redraws it. Section 8 must quote its median WITH this range, because")
    print("a single percentage would be claiming a precision the segmentation cannot carry.")
elif snd:
    print("\nLocalisation holds up across the rules too, so section 8's median can stand")
    print("on its own.")


## 9. Selection: a process-aware rung on page 8's ladder

Spec §3c. Page 8 ranks selectors that all read the final answer; this adds one that reads the
working. Among a problem's candidates, drop any containing a checker-false step, then take the
majority of what survives.

**Optional and self-skipping.** It needs page 7's candidate pile. If that is absent this reports
so and the page ships sections 7 and 8 alone, which are what it leads with anyway.

Expect a small gain, and report whatever comes out. A null result is a real finding about how
much of the failure is method rather than arithmetic, and it is one page 8's ladder is already
shaped to show.


In [ ]:
SELECTION = None

# The candidates ARE `records`, already segmented and checked in section 4, so this
# rung costs a group-by rather than a second pass over the pile. An earlier version
# re-segmented and re-checked every chain here, which doubled the run for numbers it
# already had.
cands = {}
for r in records:
    cands.setdefault(r["problem_id"], []).append(r)
cands = {k: v for k, v in cands.items() if len(v) >= 4}

if source not in ("page_07_pile", "page_09_published_traces"):
    print("the selector rung needs a pile of candidates per problem, and this run graded")
    print(f"chains from {source!r}. Sections 7 and 8 stand alone.")
elif not cands:
    print("no problem has four or more natural-length candidates; skipping. This needs")
    print("page 7's pile axis, not its split axis.")
else:
    def majority(rs):
        t = Counter(r["extracted_answer"] for r in rs if r["extracted_answer"] is not None)
        return t.most_common(1)[0][0] if t else None

    plain, filtered, emptied = [], [], 0
    for pid, rs in sorted(cands.items()):
        truth = rs[0]["ground_truth"]
        keep = [r for r in rs if r["n_false"] == 0]
        if not keep:
            emptied += 1
            keep = rs                    # dropping everything is not a selection
        plain.append(bool(grade_answer(majority(rs), truth)))
        filtered.append(bool(grade_answer(majority(keep), truth)))

    SELECTION = {
        "n_problems": len(cands),
        "config_id": CHAIN_PROVENANCE.get("page_07_config_id"),
        "page_07_run_id": CHAIN_PROVENANCE.get("page_07_run_id"),
        "mean_candidates": mean([len(v) for v in cands.values()]),
        "majority": mean(plain),
        "majority_after_step_filter": mean(filtered),
        "delta_pts": (mean(filtered) - mean(plain)) * 100,
        "n_problems_all_candidates_filtered": emptied,
        "n_problems_changed": sum(1 for a, b in zip(plain, filtered) if a != b),
    }
    harness.table(
        "selection, on page 7's pile",
        ["rule", "reads", "accuracy"],
        [["majority vote", "the answer", f"{SELECTION['majority']:.1%}"],
         ["majority after dropping chains with a false step", "the working",
          f"{SELECTION['majority_after_step_filter']:.1%}"]],
        note=f"{SELECTION['delta_pts']:+.1f} points over {len(cands)} problems, all from "
             f"{CHAIN_PROVENANCE.get('page_07_config_id')}; {emptied} had every candidate "
             f"filtered and fell back to "
             f"the full set")

    print(f"\nthe filter changed the answer on {SELECTION['n_problems_changed']} of "
          f"{len(cands)} problems")
    if abs(SELECTION["delta_pts"]) < 1 or SELECTION["n_problems_changed"] <= 2:
        print("Essentially null. Say so: at this checker's recall, reading the working")
        print("buys almost nothing at selection time, and detection and localisation are")
        print("where step grading earns its keep.")


## 10. The three specimens

The page's interactive keeps its structure: the reader marks each step, then the labels are
revealed. What changes is that the chains are real and the labels are the checker's, so the
reveal is a fact rather than an attribution to a system that never ran.

| role | what it shows |
|---|---|
| `lucky` | right answer, at least one false step. Outcome grading rewards it. |
| `broken` | wrong answer, one findable break. Outcome says *wrong*; process says *step k*. |
| `clean` | right answer, nothing false. The graders agree. |

Curated by filter, most legible first (four to eight steps, not the shortest). A role that cannot
be filled is reported rather than faked, and a missing role blocks translation for that panel: an
authored trace must not be substituted.

**Two sources, in that order, and the choice is made per role rather than per run.** Page 7's pile
first, because `lucky` and `broken` are a matched pair on one problem the page's copy names. The
easy chains of `FISH_SPECIMENS` second, for a role the pile cannot fill. `derived.json` records
which pool supplied each specimen, because one fished from levels 1-3 illustrates a population
sections 7 to 9 did not measure, and the page has to be able to chip it as such.


In [ ]:
def pick(pool):
    """The most legible specimen, which is not the shortest one.

    An earlier version took the shortest chain in range and returned a two-step trace,
    where "the first step stands" is technically the finding and pedagogically nothing.
    The mark-the-step interaction needs enough steps to be worth marking, so prefer 4 to
    8 and fall back outward rather than collapsing to the minimum.
    """
    def rank(r):
        n = r["n_steps"]
        band = 0 if 4 <= n <= 8 else (1 if 2 <= n <= 12 else 2)
        return (band, abs(n - 6), r["problem_id"])

    ranked = sorted(pool, key=rank)
    return ranked[0] if ranked else None


def specimen_steps(steps):
    """The steps as the page will read them.

    The PRM's per-step scores are stripped unless section 6's gate passed. Leaving
    them in would put a label on the reveal that this run just decided it could not
    stand behind, which is the exact defect the refactor exists to remove, only with a
    real model instead of a hand-written boolean. The raw scores stay in
    `traces.jsonl`, because suppressing a label is not the same as hiding the evidence.
    """
    drop = () if PRM_PUBLISHABLE else ("prm", "prm_label")
    return [{k: v for k, v in s.items() if k not in drop} for s in steps]


SPECIMENS, missing, SPECIMEN_SOURCES = {}, [], {}
# Where each worked example comes from, in priority order, decided per ROLE.
#
# Page 7's pile first, pinned for the pair. `lucky` and `broken` are one problem, the
# GCD of 3339, 2961, 1491 (number_theory/357): one chain makes a slip in a Euclidean
# division and still lands on 21, the other makes it and lands on 3. Same division,
# opposite luck, and one-line steps a reader can mark by hand rather than the
# brute-force enumeration this page used to show. The page's copy names that problem
# and that slip, so the pile outranks the fishing pass.
#
# Per ROLE rather than per RUN, and that is a fix rather than a refinement. An earlier
# version switched all three pools the moment fishing returned a single record, which
# silently replaced the pair with three unrelated easy problems and left the page's
# prose describing a specimen that was no longer in derived.json.
#
# Either source is ILLUSTRATIVE and chipped as such, and `derived.json` now records
# which one supplied each specimen: the measured rates in sections 7-9 are the L4-5
# pile's, and a chain fished from levels 1-3 is not from that population at all.
SPECIMEN_PROBLEM = "test/number_theory/357.json"
SPEC_SOURCES = {
    "page_07_pile_pinned": {
        "lucky":  [r for r in right if r["problem_id"] == SPECIMEN_PROBLEM
                   and r["n_false"] > 0],
        "broken": [r for r in wrong if r["problem_id"] == SPECIMEN_PROBLEM
                   and r["n_false"] == 1 and r["n_steps"] > 1],
        "clean":  [r for r in right if r["n_false"] == 0 and r["n_checkable"] > 0],
    },
    # Held to FISH_MAX_STEPS, which is the only thing this pool has over the pile. A
    # fished chain longer than that is no more legible than page 7's and there is no
    # reason to prefer it.
    "fished_easy_chains": {
        "lucky":  [r for r in fish_records if r["outcome_correct"] and r["n_false"] > 0
                   and r["n_steps"] <= FISH_MAX_STEPS],
        "broken": [r for r in fish_records if not r["outcome_correct"]
                   and r["n_false"] == 1 and 1 < r["n_steps"] <= FISH_MAX_STEPS],
        "clean":  [r for r in fish_records if r["outcome_correct"] and r["n_false"] == 0
                   and r["n_checkable"] > 0 and r["n_steps"] <= FISH_MAX_STEPS],
    },
}

for role in ("lucky", "broken", "clean"):
    got, from_source = None, None
    for name, pools in SPEC_SOURCES.items():
        got = pick(pools[role])
        if got is not None:
            from_source = name
            break
    if got is None:
        missing.append(role)
        continue
    SPECIMENS[role] = {k: got[k] for k in (
        "problem_id", "level", "question", "ground_truth", "extracted_answer",
        "outcome_correct", "trace", "tokens", "first_false", "n_false")}
    SPECIMENS[role]["steps"] = specimen_steps(got["steps"])
    SPECIMENS[role]["role"] = role
    SPECIMENS[role]["source"] = from_source
    SPECIMEN_SOURCES[role] = from_source
    SPECIMENS[role]["delimiter"] = got.get("delimiter")
    SPECIMENS[role]["prm_scored"] = bool(got.get("prm_ok", False))
    SPECIMENS[role]["prm_labels_published"] = bool(got.get("prm_ok", False)
                                                   and PRM_PUBLISHABLE)
    marks = " ".join({"true": "+", "false": "X", "unknown": "?"}[s["check"]] for s in got["steps"])
    print(f"{role:<7} {from_source:<20} {got['problem_id']:<30} L{got['level']} "
          f"{got['n_steps']:>2} steps   {marks}")

print("\n  + checked true    X checked false    ? no numeric claim to check")
print(f"  step labels on the page come from: the arithmetic checker"
      + (f" and {PRM_MODEL}" if PRM_PUBLISHABLE else " only"))

if missing:
    print(f"\nMISSING ROLES: {missing}. derived.json records this. A role with no specimen")
    print("cannot be translated to the page; do not substitute an authored trace for it.")

if "broken" in SPECIMENS:
    sp = SPECIMENS["broken"]
    print(f"\n--- {sp['problem_id']}: the localisation case " + "-" * 28)
    print(textwrap.fill("Q: " + sp["question"], 92))
    for i, s in enumerate(sp["steps"]):
        arrow = "->" if s["check"] == "false" else "  "
        print(f"{arrow} {i+1}. [{s['check']:<7}] {s['text'][:84]}")
    print(f"\noutcome grader: WRONG (said {sp['extracted_answer']}, truth {sp['ground_truth']}),")
    print(f"                which condemns all {len(sp['steps'])} steps")
    print(f"process grader: step {sp['first_false'] + 1}; the {sp['first_false']} before it stand")

## 11. Artifacts

`derived.json` is the contract with the page: it reads that and nothing else, and no inference
runs at demonstration time.

The manifest records four things the other pages do not need:

- **Which grader produced which label.** A reader disputing a step label has to be able to find
  out who made it.
- **How well the model-based grader did** on the steps we could check independently, and whether
  it cleared the gate. If it did not, its labels are stripped from the specimens rather than
  footnoted, and `prm_reported_on_page` says so.
- **The chains' real provenance**, meaning page 7's run id, configuration, temperature and
  prompt when they are page 7's, and this notebook's when they are not. Two sources with two
  decoding regimes cannot share one description.
- **What the segmentation choice was worth**, so the page can quote a number knowing how much of
  it is the chains and how much is where we cut them.

In [ ]:
manifest = {
    "run_id": f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
    "notebook": "page_09_outcome_vs_process.ipynb",
    "notebook_sha256": harness.file_sha256("page_09_outcome_vs_process.ipynb"),
    "page": 9,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "harness": harness.fingerprint(),
    "chains": {
        # CHAIN_PROVENANCE is the source's REAL configuration, not this notebook's
        # defaults. Reused chains carry page 7's run, config, temperature and prompt;
        # chains generated here carry this notebook's. An earlier version recorded
        # this page's prompt suffix either way, which asserted a prompt that page 7's
        # chains were never shown.
        **(CHAIN_PROVENANCE or {"source": source}),
        "n": len(records),
        "filter": "truncated is false and not forced -- a chain WE ended is not the "
                  "model's reasoning, and grading it would report our cap as its error",
        "filter_audit": FILTER_AUDIT,
        "filter_caveat": "every measurement is conditional on chains the model ended "
                         "itself, which skews easier than the full population",
        "generator": PRIMARY,
        "generator_revision": harness.repo_revision(PRIMARY),
        "chain_cap": CHAIN_CAP,
    },
    "graders": {
        "outcome": "reasoning_from_scratch.ch03.grade_answer",
        "arithmetic_checker": {
            "rule": "closed-form numeric on both sides, no symbols; true/false/unknown",
            "preflight_cases": len(PREFLIGHT),
            "known_limit": "cannot see method errors that use valid arithmetic",
        },
        "prm": ({"id": PRM_MODEL,
                 "revision": (PRM_ADOPTED["revision"] if PRM_FROM_CACHE
                              else harness.repo_revision(PRM_MODEL)),
                 "lineage": "same Qwen2.5 lineage as the generator, math-specialised "
                            "branch; NOT an independent family",
                 "loader": "transformers Qwen2Model + the checkpoint's score head; "
                           "the repo's bundled modeling code returns finite but "
                           "WRONG hidden states under this transformers version",
                 "separation_preflight": PRM_PREFLIGHT,
                 # Where the scores came from. A REPLAY that adopted them says so
                 # here, names the run that computed them, and flags the preflight
                 # as carried forward rather than re-run, so "this file made these
                 # numbers" stays checkable for the half that was not recomputed.
                 "scores": "adopted" if PRM_FROM_CACHE else "computed by this run",
                 "scores_adopted_from": PRM_ADOPTED,
                 "separation_preflight_is_adopted": bool(PRM_FROM_CACHE),
                 "quantisation": (PRM_ADOPTED["quantisation"] if PRM_FROM_CACHE
                                  else PRM_QUANT or str(harness.pick_dtype(get_device()))),
                 "quantisation_note": "8-bit was required to fit a 7.6B grader on a 16 GB "
                                      "card; it moves the scores and so is part of what "
                                      "section 6 measured",
                 "threshold": PRM_THRESHOLD,
                 "threshold_source": RELIABILITY["threshold_source"] if RELIABILITY
                                     else "unfitted",
                 "reliability_vs_checker": RELIABILITY,
                 "reported_on_page": PRM_PUBLISHABLE,
                 "chains_not_scored": dict(PRM_FAILURES)}
                if (PRM_READY or PRM_FROM_CACHE) else None),
    },
    "items": {"source": "MATH-500 (ch03 copy)", "filter": "level in {4, 5}",
              "n": len(items), "item_set_sha256": ITEM_SET_HASH, "seed": SEED,
              "same_as_page_7": SAME_POPULATION,
              "same_as_page_7_evidence": POPULATION_EVIDENCE},
    "step_segmentation": {"via": "harness.step_spans", "min_step_chars": MIN_STEP_CHARS,
                          "rule": "cascade: blank lines, then lines, then sentences",
                          "delimiter_mix": dict(DELIMITER_MIX),
                          "sensitivity": SEGMENTATION_CHECK},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

# The share-sound figure under each segmentation rule, so the page can quote its
# range without recomputing anything. Section 8b measured it; this only collects it.
_sound = [v["median_share_sound"] for v in SEGMENTATION_CHECK.values()
          if isinstance(v, dict) and v.get("median_share_sound") is not None]

derived = {
    "manifest": manifest,
    # Read before any number on this page. The notebook decides what may be claimed;
    # the page reads that decision and refuses to build if a blocking one is false.
    "claims": {
        # ---- blocking: without these the page has nothing to show ----------------
        # The same 100 problems page 7 drew, so pages 6 to 9 describe one population.
        "same_population_as_page_7": SAME_POPULATION,
        # Every graded chain ended itself. A chain WE cut is our experiment, not the
        # model's reasoning, and grading one would report our cap as its error.
        "every_chain_ended_itself": bool(
            FILTER_AUDIT and FILTER_AUDIT["n_kept"] > 0
            and all(not r.get("truncated") and not r.get("forced") for r in records)),
        # There are correct chains to measure a contamination rate over at all.
        "detection_measurable": len(right) > 0,
        # Section 8 has something to plot.
        "localisation_has_a_population": len(localisable) > 0,
        # All three roles filled with real traces. A missing role may NOT be filled
        # with an authored one; that is the defect this refactor exists to remove.
        "specimens_complete": not missing,

        # ---- reported, never blocking: these change what the page SAYS -----------
        # The page's first beat. Small is a finding; zero would be a fact about the
        # checker rather than about the model.
        "detection_is_nonzero": DETECTION["base_rate"] > 0,
        # Solution B teaches that a break is one line. Under half and the page must
        # follow the data instead.
        "breaks_localise_to_one_step": bool(
            LOCALISATION.get("share_with_one_false_step", 0) >= 0.5),
        # Whether the PRM cleared section 2's pre-registered bar. False means its
        # rows are suppressed, which is itself worth a sentence.
        "prm_publishable": bool(PRM_PUBLISHABLE),
        # The reason for bringing a model in at all: it reaches method errors the
        # arithmetic checker cannot see.
        "prm_sees_more_than_the_checker": bool(
            DETECTION.get("prm_reported")
            and DETECTION.get("prm_base_rate", 0) > DETECTION["base_rate"]),
        # Detection barely moves with the segmentation rule, so it may be quoted flat.
        "detection_survives_segmentation": bool(
            SEGMENTATION_CHECK.get("detection_spread_pts", 0) <= 5),
        # Localisation is a ratio of step positions, so it moves. False here means the
        # page must quote a RANGE rather than a single percentage.
        "localisation_survives_segmentation": bool(
            (SEGMENTATION_CHECK.get("share_sound_spread_pts") or 0) <= 10),
        # A delta resting on one or two problems is noise whatever the points say.
        "step_filter_helps_selection": bool(
            SELECTION and abs(SELECTION["delta_pts"]) >= 1
            and SELECTION["n_problems_changed"] > 2),
        # Most of a chain is beyond a deterministic checker, which IS the argument
        # for section 6 and belongs on the page rather than in a footnote.
        "most_steps_are_beyond_the_checker": bool(
            lab["unknown"] / max(len(all_steps), 1) > 0.8),
    },
    # The exact figures the page quotes, collected so it never recomputes one.
    "headline": {
        "detection_checker": {
            "n_right_chains": DETECTION["n_right"],
            "n_contaminated": DETECTION["n_right_with_false_step"],
            "rate": DETECTION["base_rate"],
            "n_problems": DETECTION["n_problems_with_a_contaminated_right_chain"],
            "n_problems_with_a_right_chain": DETECTION["n_problems_with_a_right_chain"],
        },
        "detection_prm": ({"rate": DETECTION["prm_base_rate"],
                           "n_chains_scored": DETECTION["prm_n_chains"],
                           "auc": DETECTION["prm_auc"],
                           "threshold": DETECTION["prm_threshold"]}
                          if DETECTION.get("prm_reported") else None),
        "localisation": {
            "median_share_sound": LOCALISATION.get("median_share_sound"),
            # The sensitivity is a SPREAD, not a pair of bounds around the median
            # above. Section 8b runs on a seeded subsample, so its canonical row is
            # not this median and quoting its min and max as a range around this
            # number would place the range above the figure it brackets. What
            # transfers is how far the answer moves when the rule changes.
            "segmentation_spread_pts": SEGMENTATION_CHECK.get("share_sound_spread_pts"),
            "share_sound_on_subsample": {
                "n_chains": SEGMENTATION_CHECK.get("n_chains"),
                "low": min(_sound) if _sound else None,
                "high": max(_sound) if _sound else None,
            },
            "n_localisable": LOCALISATION["n_localisable"],
            "n_wrong": LOCALISATION["n_wrong"],
            "n_problems": LOCALISATION["n_problems"],
            "share_with_one_false_step": LOCALISATION.get("share_with_one_false_step"),
        },
        "selection": SELECTION,
        "step_labels": {k: lab[k] for k in ("true", "false", "unknown")},
        "n_chains": len(records),
    },
    "detection": DETECTION,
    "localisation": LOCALISATION,
    "selection": SELECTION,
    "prm_reliability": RELIABILITY,
    "prm_reported_on_page": PRM_PUBLISHABLE,
    "segmentation_sensitivity": SEGMENTATION_CHECK,
    "filter_audit": FILTER_AUDIT,
    "chain_provenance": CHAIN_PROVENANCE,
    "step_label_counts": dict(lab),
    "specimens": SPECIMENS,
    # Which pool supplied each specimen. A role filled from the fishing pass is a
    # chain from levels 1-3, which is NOT the population sections 7 to 9 measured,
    # and the page needs to be able to say so beside the reveal.
    "specimen_sources": SPECIMEN_SOURCES,
    "missing_roles": missing,
}

BLOCKING = ("same_population_as_page_7", "every_chain_ended_itself",
            "detection_measurable", "localisation_has_a_population",
            "specimens_complete")
harness.table(
    "what this run says the page may claim",
    ["claim", "blocking", "value"],
    [[k, "yes" if k in BLOCKING else "", "true" if v else "FALSE"]
     for k, v in derived["claims"].items()],
    note="the page reads this block; a blocking claim that is false is a build error "
         "there, never a conditional in the component")

# The gate runs BEFORE anything is written, and the order is the whole point. Writing
# first meant a run that failed a blocking claim had already replaced the last good
# derived.json with one the page refuses to load, so a bad run cost the previous run
# as well. A refused run now leaves the artifacts it could not improve on.
_failed = [k for k in BLOCKING if not derived["claims"][k]]
assert not _failed, (
    f"blocking claim(s) false: {_failed}. Nothing was written, so the last good run is "
    f"still on disk. This run cannot be translated to the page; fix the run rather "
    f"than the gate.")
print("all blocking claims hold; the reported ones are the page's copy decisions.\n")

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
with TRACES_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

for p, tier in ((DERIVED_PATH, "bundled"), (MANIFEST_PATH, "bundled"),
                (TRACES_PATH, "download")):
    print(f"{p.name:<16} {harness.kb(p):>9.1f} KB   <- {tier}")

## Notes for re-derivation

- The checker is **high precision, low recall by design**. Every `false` is verifiable by hand;
  every `unknown` is a step it declined to judge, not one it approved.
- Detection and localisation come from the checker alone, so they stand whether or not the PRM
  ran, and whether or not it cleared the gate. That separation is deliberate: the page's two
  load-bearing numbers do not depend on a model's opinion.
- The chains are filtered to those the model ended itself. That filter, not the source, is what
  makes them gradable; page 7's pile qualifies and its split axis does not. It is also a
  selection toward easier problems, and section 3 measures how much.
- **Only one of page 7's configurations is read.** Its dump holds the pile, the split axis, a
  diversity family at four other temperatures and a replicated greedy set. Pooling them would
  grade four decoding regimes and report the mixture as one population.
- **The PRM's threshold is fitted, not given.** It is chosen on the checker's labels and recorded
  as fitted, and the reliability beside it was measured on arithmetic steps, which are the
  friendliest ground the PRM will see here. Treat it as an upper bound.
- **The PRM is not an independent family.** Same Qwen2.5 lineage as the generator, math branch.
  It tests whether a bigger specialised checkpoint catches this one's errors, not whether an
  unrelated grader does.
- Reused chains were **sampled**, not greedy, so re-running page 7 with a different seed changes
  which chain each problem contributes. Chains generated here are greedy and still diverge
  across hardware on a few items. Expect the shape to reproduce, not every digit.

What the page may and may not claim is [`docs/page_9_refactor.md`](../docs/page_9_refactor.md),
which the translation step checks itself against
([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).